# causal-tracing-auto-v2

Self-contained causal tracing notebook for automatic early-site layer detection. The main body runs paired subject-last MLP-window tracing with a discovery/confirmation split and saves graphs/results. Covariance matrix computation and the ROME benchmark are separate optional stages at the end.

**Method status:** v2 implements the audited Latium rule directly: discovery chooses one full-width window by mean paired indirect effect, and held-out confirmation only tests that exact window. The configured ROME layer is graph-only and never affects trace selection.


## 0. Optional Install Cell

In [ ]:
# Uncomment in Colab or a fresh environment.
# %pip install -q torch transformers datasets accelerate pandas matplotlib tqdm
# Optional for quantized loading on limited VRAM:
# %pip install -q bitsandbytes

## 1. Imports

In [ ]:
from __future__ import annotations

from contextlib import contextmanager
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
import gc
import json
import math
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import DatasetDict, concatenate_datasets, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.grid'] = True
print('Imports OK')

## 2. Batch Settings

In [ ]:
# Run one or many. Example: ['qwen3-4b', 'qwen3-8b']
MODEL_CONFIGS = ['gpt2-xl', 'qwen3-8b', 'mistral-7b-v0.3', 'opt-6.7b', 'falcon-7b', 'granite4-micro', 'deepseek-7b-base']

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'bf16'  # bf16, f16, f32, auto
USE_DEVICE_MAP_AUTO = False
TRUST_REMOTE_CODE = True
LOAD_IN_4BIT = False
ADAPTER_VALIDATE_ALL_LAYERS = True

DATASET_NAME = 'azhx/counterfact'
DATASET_SPLITS = ['train', 'test']
NUM_VALID_FACTS = 100
# Per-model valid-fact caps prevent one 7B/8B model from blocking the whole batch.
# Set a model to 100 here when you want the full robust run for that model.
MODEL_VALID_FACT_LIMITS = {
    'gpt2-xl': 100,
    'qwen3-8b': 20,
    'mistral-7b-v0.3': 100,
    'opt-6.7b': 20,
    'falcon-7b': 100,
    'granite4-micro': 100,
    'deepseek-7b-base': 20,
}
DISCOVERY_FRACTION = 0.5
MAX_DATASET_EXAMPLES_TO_SCAN = 10000
MODEL_MAX_DATASET_EXAMPLES_TO_SCAN = {}
TRACE_STATUS_EVERY = 25
TRACE_MAX_SECONDS_PER_MODEL = None
MODEL_TRACE_MAX_SECONDS = {
    'qwen3-8b': 1800,
    'mistral-7b-v0.3': 1800,
    'opt-6.7b': 1800,
    'falcon-7b': None,
    'granite4-micro': None,
    'deepseek-7b-base': 1800,
}
MIN_FACTS_AFTER_TRACE_TIMEOUT = 8
NUM_NOISE_SAMPLES = 10
NOISE_BATCH_SIZE = 2
NOISE_MULTIPLIER = 3.0
SEED = 42

WINDOW_MODE = 'canonical_rome'  # canonical_rome or proportional
WINDOW_SIZE = 10
WINDOW_FRACTION = 0.20

REQUIRE_CORRECT_CLEAN = True
MIN_TOTAL_EFFECT = 0.03

# Technical floor only. Use enough valid facts to make the held-out interval meaningful.
MIN_CONFIRMATION_FACTS = 2
BOOTSTRAP_SAMPLES = 1000
CONFIDENCE_LEVEL = 0.95

# Graph-only config-layer reference markers. These never affect tracing, region selection, representative centers, or saved causal selection.
# Values mirror src/config/model/*.yaml `layer:` entries for visual comparison only.
CONFIG_REFERENCE_LAYERS = {
    'gpt2-medium': 8,
    'gpt2-large': 12,
    'gpt2-xl': 17,
    'gpt-j-6b': 5,
    'falcon-7b': 5,
    'opt-6.7b': 15,
    'llama2-7b': 6,
    'mistral-7b-v0.1': 5,
    'mistral-7b-v0.3': 17,
    'deepseek-7b-base': 20,
    'deepseek-r1-llama3-8b': 0,
    'qwen2.5-1.5b': 7,
    'qwen3-0.6b': 5,
    'qwen3-1.7b': 9,
    'qwen3-4b': 6,
    'qwen3-8b': 7,
    'granite4-micro': 35,
    'qwen3-guard-0.6b': 5,
}
for alias in ['qwen3-8b-prefixtest-external-long', 'qwen3-8b-prefixtest-external-medium', 'qwen3-8b-prefixtest-external-short', 'qwen3-8b-prefixtest-self-long', 'qwen3-8b-prefixtest-self-medium', 'qwen3-8b-prefixtest-self-short', 'qwen3-8b-prefixtest-template-long', 'qwen3-8b-prefixtest-template-medium', 'qwen3-8b-prefixtest-template-short']:
    CONFIG_REFERENCE_LAYERS[alias] = CONFIG_REFERENCE_LAYERS['qwen3-8b']

SAVE = True
OUT_ROOT = Path('./analysis_out/causal_tracing_auto_v2')
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

# Main tracing gate and optional end-stage gates.
RUN_CAUSAL_TRACE = True
RUN_SECOND_MOMENT = True  # optional covariance cell at the end
RUN_ROME_BENCHMARK = True  # optional ROME benchmark cell at the end
ROME_BENCHMARK_NUM_EDITS = 100
ROME_BENCHMARK_START_IDX = 0
ROME_BENCHMARK_MODELS = MODEL_CONFIGS
ROME_OPTIMIZER_VERBOSE = False

# ROME/covariance benchmark layer policy. The config layer is never used for selection.
# Only held-out-confirmed trace centers proceed as new layers; config fallback is benchmark-only.
ROME_RETRY_CONFIG_LAYER_ON_ZERO_EVALUATED = True  # benchmark fallback only; does not affect causal trace selection
ROME_RUN_CONFIG_LAYER_ON_TRACE_SKIP = True  # if tracing skips because selected layer matches config, still benchmark the config layer
ROME_FINAL_TARGET_EVALUATED_EDITS = True  # final ROME benchmark tries to collect ROME_BENCHMARK_NUM_EDITS evaluated edits
ROME_MAX_ATTEMPT_MULTIPLIER = 5  # scan up to this many dataset rows per requested evaluated edit
ROME_PRIMARY_ZERO_EVAL_ABORT_AFTER = 20  # stop a failing primary layer early and retry config/reference layer

# ROME covariance / second-moment settings. ROME consumes the inverse covariance matrix,
# and this notebook also saves the raw covariance matrix for audit/reuse.
SECOND_MOMENT_ALLOW_AUTOCOMPUTE = False
SECOND_MOMENT_TARGET_SAMPLES = 100_000
SECOND_MOMENT_DIR = Path('./data/second_moment_stats')
SAVE_RAW_COVARIANCE = True
RAW_COVARIANCE_DIR = SECOND_MOMENT_DIR / 'raw_covariance'
SECOND_MOMENT_DATASET_NAME = 'Salesforce/wikitext'
SECOND_MOMENT_DATASET_CONFIG = 'wikitext-103-raw-v1'
SECOND_MOMENT_DATASET_SPLITS = ['train', 'test', 'validation']
SECOND_MOMENT_BATCH_SIZE_MODE = 'auto'  # mirrors src/rome/common.py: auto/dynamic/manual
SECOND_MOMENT_BATCH_SIZE = None  # set an int to override auto sizing
SECOND_MOMENT_MAX_LENGTH = None
SECOND_MOMENT_MIN_TEXT_LENGTH = 50
SECOND_MOMENT_CLEAR_CACHE_EVERY = 0

# Ensure covariance output directories exist even on a fresh checkout/Colab runtime.
SECOND_MOMENT_DIR.mkdir(parents=True, exist_ok=True)
if SAVE_RAW_COVARIANCE:
    RAW_COVARIANCE_DIR.mkdir(parents=True, exist_ok=True)

if not 0 < DISCOVERY_FRACTION < 1:
    raise ValueError('DISCOVERY_FRACTION must be strictly between 0 and 1')
if MIN_CONFIRMATION_FACTS < 2:
    raise ValueError('MIN_CONFIRMATION_FACTS must be at least 2')
if NUM_NOISE_SAMPLES <= 0 or NOISE_BATCH_SIZE <= 0 or BOOTSTRAP_SAMPLES <= 0:
    raise ValueError('Noise sample, noise batch, and bootstrap counts must be positive')
if not 0 < CONFIDENCE_LEVEL < 1:
    raise ValueError('CONFIDENCE_LEVEL must be strictly between 0 and 1')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 3. Built-In Model Presets

In [ ]:
MODEL_PRESETS = {
    'gpt2-medium': {'name': 'gpt2-medium', 'layers': 24, 'embedding': 'transformer.wte', 'block_template': 'transformer.h.{}', 'mlp_template': 'transformer.h.{}.mlp.c_proj', 'dtype': 'bf16'},
    'gpt2-large': {'name': 'gpt2-large', 'layers': 36, 'embedding': 'transformer.wte', 'block_template': 'transformer.h.{}', 'mlp_template': 'transformer.h.{}.mlp.c_proj', 'dtype': 'bf16'},
    'gpt2-xl': {'name': 'gpt2-xl', 'layers': 48, 'embedding': 'transformer.wte', 'block_template': 'transformer.h.{}', 'mlp_template': 'transformer.h.{}.mlp.c_proj', 'dtype': 'bf16'},
    'gpt-j-6b': {'name': 'EleutherAI/gpt-j-6B', 'layers': 28, 'embedding': 'transformer.wte', 'block_template': 'transformer.h.{}', 'mlp_template': 'transformer.h.{}.mlp.fc_out', 'dtype': 'bf16'},
    'falcon-7b': {'name': 'tiiuae/falcon-7b', 'layers': 32, 'embedding': 'transformer.word_embeddings', 'block_template': 'transformer.h.{}', 'mlp_template': 'transformer.h.{}.mlp.dense_4h_to_h', 'dtype': 'bf16', 'trust_remote_code': False},
    'opt-6.7b': {'name': 'facebook/opt-6.7b', 'layers': 32, 'embedding': 'model.decoder.embed_tokens', 'block_template': 'model.decoder.layers.{}', 'mlp_template': 'model.decoder.layers.{}.fc2', 'dtype': 'bf16'},
    'llama2-7b': {'name': 'NousResearch/Llama-2-7b-hf', 'layers': 32, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'mistral-7b-v0.1': {'name': 'mistralai/Mistral-7B-v0.1', 'layers': 32, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'mistral-7b-v0.3': {'name': 'mistralai/Mistral-7B-v0.3', 'layers': 32, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'deepseek-7b-base': {'name': 'deepseek-ai/deepseek-llm-7b-base', 'layers': 30, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'deepseek-r1-llama3-8b': {'name': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B', 'layers': 32, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'qwen2.5-1.5b': {'name': 'Qwen/Qwen2.5-Math-1.5B', 'layers': 28, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'qwen3-0.6b': {'name': 'Qwen/Qwen3-0.6B', 'layers': 28, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'qwen3-1.7b': {'name': 'Qwen/Qwen3-1.7B', 'layers': 28, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'qwen3-4b': {'name': 'Qwen/Qwen3-4B', 'layers': 36, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'qwen3-8b': {'name': 'Qwen/Qwen3-8B', 'layers': 36, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'granite4-micro': {'name': 'ibm-granite/granite-4.0-micro', 'layers': 40, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.shared_mlp.output_linear', 'dtype': 'bf16'},
    'qwen3-guard-0.6b': {'name': 'Qwen/Qwen3Guard-Gen-0.6B', 'layers': 28, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
}
for alias in ['qwen3-8b-prefixtest-external-long', 'qwen3-8b-prefixtest-external-medium', 'qwen3-8b-prefixtest-external-short', 'qwen3-8b-prefixtest-self-long', 'qwen3-8b-prefixtest-self-medium', 'qwen3-8b-prefixtest-self-short', 'qwen3-8b-prefixtest-template-long', 'qwen3-8b-prefixtest-template-medium', 'qwen3-8b-prefixtest-template-short']:
    MODEL_PRESETS[alias] = dict(MODEL_PRESETS['qwen3-8b'])

unknown = [name for name in MODEL_CONFIGS if name not in MODEL_PRESETS]
if unknown:
    raise ValueError(f'Unknown MODEL_CONFIGS={unknown}. Available: {sorted(MODEL_PRESETS)}')
print('Batch models:', MODEL_CONFIGS)

## 4. Data Types

In [ ]:
@dataclass
class WindowTrace:
    center: int
    start: int
    end: int
    layers: list[int]
    module_names: list[str]
    is_full_width: bool
    restore_probabilities: np.ndarray
    ie_samples: np.ndarray
    mean_ie: float
    median_ie: float
    std_ie: float
    sem_ie: float
    normalized_recovery: float

@dataclass
class FactTrace:
    prompt_idx: int
    prompt: str
    subject: str
    target_full_text: str
    target_first_token_id: int
    target_first_token_text: str
    target_num_tokens: int
    clean_top_token: str
    clean_probability: float
    clean_top_probability: float
    corrupt_probabilities: np.ndarray
    total_effect: float
    corrupt_relative_std: float
    subject_positions: list[int]
    subject_tokens: list[str]
    subject_last_position: int
    subject_last_token: str
    prompt_last_position: int
    prompt_last_token: str
    windows: list[WindowTrace]

    @property
    def mean_corrupt_probability(self):
        return float(np.mean(self.corrupt_probabilities))

class TraceSkip(Exception):
    def __init__(self, reason, detail):
        super().__init__(detail)
        self.reason = reason
        self.detail = detail

## 5. Shared Helpers

In [ ]:
def normalize_counterfact_dataset(raw):
    parts = []
    if isinstance(raw, DatasetDict):
        for split in DATASET_SPLITS:
            if split in raw:
                parts.append(raw[split])
        if not parts:
            parts = [next(iter(raw.values()))]
        return concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    return raw

@contextmanager
def temporary_hooks(hooks):
    handles = []
    try:
        for module, hook in hooks:
            handles.append(module.register_forward_hook(hook))
        yield
    finally:
        for handle in handles:
            handle.remove()

def hidden_from_output(output):
    return output[0] if isinstance(output, tuple) else output

def replace_hidden(output, hidden):
    if isinstance(output, tuple):
        values = list(output)
        values[0] = hidden
        return tuple(values)
    return hidden

def repeat_inputs(inputs, repeats: int):
    return {key: value.repeat((repeats,) + (1,) * (value.dim() - 1)) for key, value in inputs.items() if torch.is_tensor(value)}

def bootstrap_ci(matrix, samples=BOOTSTRAP_SAMPLES, confidence=CONFIDENCE_LEVEL, seed=SEED):
    matrix = np.asarray(matrix)
    if matrix.shape[0] == 1:
        return matrix[0].copy(), matrix[0].copy()
    rng = np.random.default_rng(int(seed))
    boot = np.empty((int(samples), matrix.shape[1]), dtype=np.float64)
    n = matrix.shape[0]
    for sample_idx in range(int(samples)):
        idx = rng.integers(0, n, size=n)
        boot[sample_idx] = matrix[idx].mean(axis=0)
    alpha = (1.0 - float(confidence)) / 2.0
    return np.quantile(boot, alpha, axis=0), np.quantile(boot, 1.0 - alpha, axis=0)

def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    return value

## 6. Load Dataset Once

In [ ]:
print(f'Loading dataset {DATASET_NAME}...')
raw_dataset = load_dataset(DATASET_NAME)
dataset = normalize_counterfact_dataset(raw_dataset)
print(dataset)

## 7. Batch Runner

In [ ]:
def run_one_model(model_config: str, dataset):
    preset = MODEL_PRESETS[model_config]
    out_dir = OUT_ROOT / f'{model_config}_{RUN_TIMESTAMP}'
    dtype_picker = {'auto': 'auto', 'bf16': torch.bfloat16, 'f16': torch.float16, 'f32': torch.float32}
    dtype = dtype_picker.get(DTYPE, dtype_picker.get(preset.get('dtype', 'auto'), 'auto'))
    model_name = preset['name']
    model_trust_remote_code = bool(preset.get('trust_remote_code', TRUST_REMOTE_CODE))

    def ensure_padding(tok):
        if tok.pad_token is None:
            if tok.eos_token is not None:
                tok.pad_token = tok.eos_token
            elif tok.eos_token_id is not None:
                tok.pad_token_id = tok.eos_token_id
        if tok.pad_token_id is None and tok.eos_token_id is not None:
            tok.pad_token_id = tok.eos_token_id
        return tok

    model_kwargs = {'trust_remote_code': model_trust_remote_code}
    if dtype != 'auto':
        model_kwargs['torch_dtype'] = dtype
    if os.environ.get('HF_TOKEN'):
        model_kwargs['token'] = os.environ['HF_TOKEN']
    if USE_DEVICE_MAP_AUTO:
        model_kwargs['device_map'] = 'auto'
    if LOAD_IN_4BIT:
        model_kwargs['load_in_4bit'] = True
        model_kwargs['device_map'] = 'auto'

    print(f'\n===== Loading {model_config}: {model_name} (trust_remote_code={model_trust_remote_code}) =====')
    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    if not USE_DEVICE_MAP_AUTO and not LOAD_IN_4BIT:
        model = model.to(DEVICE)
    model.eval()
    tok_kwargs = {'trust_remote_code': model_trust_remote_code}
    if os.environ.get('HF_TOKEN'):
        tok_kwargs['token'] = os.environ['HF_TOKEN']
    tokenizer = ensure_padding(AutoTokenizer.from_pretrained(model_name, **tok_kwargs))
    num_layers = int(getattr(model.config, 'num_hidden_layers', preset.get('layers')))
    input_device = next(model.parameters()).device
    primary_window_size = max(1, int(round(num_layers * WINDOW_FRACTION))) if WINDOW_MODE == 'proportional' else int(WINDOW_SIZE)
    if not 1 <= primary_window_size <= num_layers:
        raise ValueError(f'Window size must be between 1 and {num_layers}, got {primary_window_size}')
    config_reference_layer = CONFIG_REFERENCE_LAYERS.get(model_config)
    print(f'input_device={input_device}; num_layers={num_layers}; window_size={primary_window_size}; config_reference_layer={config_reference_layer}')

    available_module_names = {name for name, _module in model.named_modules()}

    def resolve_module(name: str):
        for module_name, module in model.named_modules():
            if module_name == name:
                return module
        raise KeyError(f'Module not found: {name}')

    def embedding_module_name():
        name = preset['embedding']
        if name not in available_module_names:
            raise KeyError(f'Embedding module {name!r} not found')
        return name

    def candidate_mlp_output_module_names(layer: int):
        primary = preset['mlp_template'].format(int(layer))
        candidates = [primary]
        for marker in ('.mlp.', '.shared_mlp.'):
            if marker in primary:
                candidates.append(primary.split(marker, 1)[0] + marker.rstrip('.'))
        block = preset.get('block_template', '').format(int(layer)) if preset.get('block_template') else ''
        if block:
            candidates.extend([f'{block}.mlp', f'{block}.shared_mlp', f'{block}.feed_forward', f'{block}.ffn', f'{block}.fc2', f'{block}.mlp.dense_4h_to_h'])
        candidates.extend([f'transformer.h.{int(layer)}.mlp', f'model.layers.{int(layer)}.mlp', f'model.layers.{int(layer)}.shared_mlp', f'model.decoder.layers.{int(layer)}.fc2'])
        unique = []
        for name in candidates:
            if name and name not in unique:
                unique.append(name)
        return unique

    def mlp_output_module_name(layer: int):
        for name in candidate_mlp_output_module_names(layer):
            if name in available_module_names:
                return name
        raise KeyError(f'Could not resolve MLP-output module for layer {layer}. Tried {candidate_mlp_output_module_names(layer)[:8]}')

    module_map = pd.DataFrame([{'layer': layer, 'mlp_output_module': mlp_output_module_name(layer)} for layer in range(num_layers)])

    def tokenize_prompt(prompt_text: str):
        return tokenizer(prompt_text, return_tensors='pt', padding=False).to(input_device)

    def window_layers(center: int):
        left_width = primary_window_size // 2
        right_width = primary_window_size - left_width
        start = max(0, int(center) - left_width)
        end = min(int(num_layers), int(center) + right_width)
        return list(range(start, end))

    def window_metadata(center: int):
        layers = window_layers(center)
        return {'window_center': int(center), 'window_start': int(layers[0]), 'window_end': int(layers[-1] + 1), 'window_size_actual': int(len(layers)), 'window_layers': layers, 'window_is_full_width': len(layers) == primary_window_size, 'excluded_from_ranking': len(layers) != primary_window_size, 'exclusion_reason': None if len(layers) == primary_window_size else 'partial_boundary_window'}

    def target_token_info(target: str):
        ids = tokenizer(f' {target.strip()}', add_special_tokens=False)['input_ids']
        if ids and isinstance(ids[0], list):
            ids = ids[0]
        if not ids:
            ids = tokenizer(target, add_special_tokens=False)['input_ids']
        bos = getattr(tokenizer, 'bos_token_id', None)
        if bos is not None and len(ids) > 1 and ids[0] == bos:
            ids = ids[1:]
        if not ids:
            raise ValueError(f'Could not tokenize target {target!r}')
        return int(ids[0]), tokenizer.decode([int(ids[0])]), len(ids)

    def subject_span_from_offsets(prompt: str, subject: str):
        starts = []
        cursor = 0
        while True:
            idx = prompt.find(subject, cursor)
            if idx == -1:
                break
            starts.append(idx)
            cursor = idx + max(1, len(subject))
        if len(starts) != 1:
            raise ValueError(f'Subject span ambiguous or missing: found {len(starts)} matches')
        char_start = starts[0]
        char_end = char_start + len(subject)
        encoded = tokenizer(prompt, return_offsets_mapping=True, return_tensors='pt')
        positions = []
        for idx, (start, end) in enumerate(encoded['offset_mapping'][0].detach().cpu().tolist()):
            if end <= start:
                continue
            if end > char_start and start < char_end:
                positions.append(int(idx))
        if not positions:
            raise ValueError('Subject span could not be mapped to tokens')
        return positions

    def decode_position(input_ids, position: int):
        return tokenizer.decode([int(input_ids[0, int(position)].detach().cpu().item())])

    def make_noise(num_samples, subject_len, hidden_size, noise_std, device, model_dtype, seed):
        gen = torch.Generator(device='cpu')
        gen.manual_seed(int(seed))
        noise = torch.randn((num_samples, subject_len, hidden_size), generator=gen, dtype=torch.float32)
        return (noise * float(noise_std)).to(device=device, dtype=model_dtype)

    def corrupt_hook(subject_positions, noise_samples):
        positions = [int(pos) for pos in subject_positions]
        def hook(_module, _inputs, output):
            hidden = hidden_from_output(output)
            changed = hidden.clone()
            noise = noise_samples.to(device=changed.device, dtype=changed.dtype)
            for offset, token_idx in enumerate(positions):
                changed[:, token_idx, :] = changed[:, token_idx, :] + noise[:, offset, :]
            return replace_hidden(output, changed)
        return hook

    def mlp_state_at_position(hidden: torch.Tensor, position: int, sequence_length: int):
        position = int(position)
        sequence_length = int(sequence_length)
        if hidden.dim() == 3:
            if hidden.shape[1] <= position:
                raise RuntimeError(f'MLP output sequence length {hidden.shape[1]} does not contain position {position}')
            return hidden[0, position, :].detach().clone()
        if hidden.dim() == 2:
            if hidden.shape[0] % sequence_length != 0:
                raise RuntimeError(f'2D MLP output first dimension {hidden.shape[0]} is not divisible by sequence length {sequence_length}')
            row = position
            if row >= hidden.shape[0]:
                raise RuntimeError(f'2D MLP output shape {tuple(hidden.shape)} does not contain position {position}')
            return hidden[row, :].detach().clone()
        raise RuntimeError(f'Unsupported MLP output rank {hidden.dim()} with shape {tuple(hidden.shape)}')

    def patch_mlp_position(hidden: torch.Tensor, position: int, clean_state: torch.Tensor, sequence_length: int):
        position = int(position)
        sequence_length = int(sequence_length)
        changed = hidden.clone()
        state = clean_state.to(device=changed.device, dtype=changed.dtype)
        if hidden.dim() == 3:
            changed[:, position, :] = state
            return changed
        if hidden.dim() == 2:
            if hidden.shape[0] % sequence_length != 0:
                raise RuntimeError(f'2D MLP output first dimension {hidden.shape[0]} is not divisible by sequence length {sequence_length}')
            batch_count = hidden.shape[0] // sequence_length
            rows = torch.arange(batch_count, device=changed.device, dtype=torch.long) * sequence_length + position
            changed[rows, :] = state
            return changed
        raise RuntimeError(f'Unsupported MLP output rank {hidden.dim()} with shape {tuple(hidden.shape)}')

    def supported_mlp_output_shape(shape, sequence_length: int):
        if shape is None:
            return False
        if len(shape) == 3:
            return shape[0] == 1 and shape[1] == int(sequence_length)
        if len(shape) == 2:
            return shape[0] % int(sequence_length) == 0
        return False

    def restore_position_hook(position: int, clean_state: torch.Tensor, sequence_length: int):
        def hook(_module, _inputs, output):
            hidden = hidden_from_output(output)
            changed = patch_mlp_position(hidden, position, clean_state, sequence_length)
            return replace_hidden(output, changed)
        return hook

    def token_probability(logits, token_id: int):
        return torch.softmax(logits[:, -1, :], dim=-1)[:, int(token_id)].detach().float().cpu().numpy()

    def mlp_shape_mode(shape, sequence_length: int):
        if shape is None:
            return 'missing'
        if len(shape) == 3 and shape[0] == 1 and shape[1] == int(sequence_length):
            return 'batch_seq_hidden'
        if len(shape) == 2 and shape[0] % int(sequence_length) == 0:
            return 'flat_batch_seq_hidden'
        return 'unsupported'

    def validate_mlp_adapter_shapes():
        # Adapter validation uses a neutral prompt and never reads configured/reference layers for selection.
        probe_inputs = tokenize_prompt('The capital of France is')
        probe_sequence_length = int(probe_inputs['input_ids'].shape[1])
        layers_to_probe = range(num_layers) if ADAPTER_VALIDATE_ALL_LAYERS else [num_layers // 2]
        captured = {}
        hooks = []
        for layer in layers_to_probe:
            name = mlp_output_module_name(int(layer))
            def make_hook(layer_idx, module_name):
                def hook(_module, _inputs, output):
                    hidden = hidden_from_output(output)
                    captured[int(layer_idx)] = {'shape': tuple(hidden.shape), 'module_name': module_name}
                    return output
                return hook
            hooks.append((resolve_module(name), make_hook(int(layer), name)))
        with torch.inference_mode(), temporary_hooks(hooks):
            model(**probe_inputs, use_cache=False)
        rows = []
        for layer in layers_to_probe:
            layer = int(layer)
            item = captured.get(layer, {'shape': None, 'module_name': mlp_output_module_name(layer)})
            shape = item['shape']
            mode = mlp_shape_mode(shape, probe_sequence_length)
            if mode in {'missing', 'unsupported'}:
                raise RuntimeError(
                    f'Adapter validation failed for {model_config} layer {layer} module {item["module_name"]}: '
                    f'captured MLP output shape {shape}; expected 3D [batch, seq, hidden] or 2D [batch*seq, hidden]'
                )
            rows.append({'layer': layer, 'mlp_output_module': item['module_name'], 'probe_shape': shape, 'shape_mode': mode})
        df = pd.DataFrame(rows)
        modes = ', '.join(f'{mode}:{count}' for mode, count in df['shape_mode'].value_counts().sort_index().items())
        print(f'Adapter validation OK: {len(df)} layer(s), shape modes: {modes}')
        return df

    adapter_validation = validate_mlp_adapter_shapes()
    module_map = module_map.merge(adapter_validation, on=['layer', 'mlp_output_module'], how='left')

    def cache_clean_mlp_outputs(inputs, position: int):
        captured = {}
        hooks = []
        chosen_names = []
        sequence_length = int(inputs['input_ids'].shape[1])
        for layer in range(num_layers):
            name = mlp_output_module_name(layer)
            chosen_names.append(name)
            def make_hook(layer_idx):
                def hook(_module, _inputs, output):
                    hidden = hidden_from_output(output)
                    captured[layer_idx] = mlp_state_at_position(hidden, position, sequence_length)
                    return output
                return hook
            hooks.append((resolve_module(name), make_hook(layer)))
        with torch.inference_mode(), temporary_hooks(hooks):
            model(**inputs, use_cache=False)
        return {'position': int(position), 'sequence_length': sequence_length, 'module_names': chosen_names, 'states': {layer: captured[layer].detach().clone() for layer in captured}}

    def trace_mlp_windows_for_fact(inputs, target_id, subject_positions, noise_samples, corrupt_probabilities, clean_probability, restore_position):
        emb = resolve_module(embedding_module_name())
        clean_cache = cache_clean_mlp_outputs(inputs, restore_position)
        windows = []
        noise_batch_size = max(1, min(int(NOISE_BATCH_SIZE), int(NUM_NOISE_SAMPLES)))
        with torch.inference_mode():
            for center in range(num_layers):
                meta = window_metadata(center)
                restore_probs = np.zeros((NUM_NOISE_SAMPLES,), dtype=np.float32)
                module_names_for_window = [clean_cache['module_names'][layer] for layer in meta['window_layers']]
                for batch_start in range(0, NUM_NOISE_SAMPLES, noise_batch_size):
                    batch_end = min(NUM_NOISE_SAMPLES, batch_start + noise_batch_size)
                    repeated = repeat_inputs(inputs, batch_end - batch_start)
                    hooks = [(emb, corrupt_hook(subject_positions, noise_samples[batch_start:batch_end]))]
                    for layer in meta['window_layers']:
                        name = clean_cache['module_names'][layer]
                        hooks.append((resolve_module(name), restore_position_hook(restore_position, clean_cache['states'][layer], clean_cache['sequence_length'])))
                    with temporary_hooks(hooks):
                        restored = model(**repeated, use_cache=False)
                    restore_probs[batch_start:batch_end] = token_probability(restored.logits, target_id)
                    del repeated, restored
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                ie = restore_probs - corrupt_probabilities
                windows.append(WindowTrace(center=int(center), start=meta['window_start'], end=meta['window_end'], layers=meta['window_layers'], module_names=module_names_for_window, is_full_width=meta['window_is_full_width'], restore_probabilities=restore_probs, ie_samples=ie, mean_ie=float(np.mean(ie)), median_ie=float(np.median(ie)), std_ie=float(np.std(ie)), sem_ie=float(np.std(ie) / max(np.sqrt(len(ie)), 1.0)), normalized_recovery=float(np.mean(ie) / max(abs(clean_probability - float(np.mean(corrupt_probabilities))), 1e-9))))
        return windows

    def row_to_fact(row):
        rr = row.get('requested_rewrite', row)
        subject = rr['subject']
        target = rr['target_true']['str'] if isinstance(rr.get('target_true'), dict) else rr.get('target')
        prompt = rr['prompt'].format(subject)
        return prompt, subject, target

    def trace_fact(prompt_idx: int, prompt: str, subject: str, target: str):
        inputs = tokenize_prompt(prompt)
        subject_positions = subject_span_from_offsets(prompt, subject)
        subject_last = int(subject_positions[-1])
        prompt_last = int(inputs['input_ids'].shape[1] - 1)
        target_id, target_first_text, target_num_tokens = target_token_info(target)
        emb = resolve_module(embedding_module_name())
        embedding_std = float(emb.weight.detach().float().std().item())
        noise_std = NOISE_MULTIPLIER * embedding_std
        hidden_size = int(emb.weight.shape[1])
        with torch.inference_mode():
            clean = model(**inputs, use_cache=False)
            probs = torch.softmax(clean.logits[:, -1, :], dim=-1)[0]
            clean_probability = float(probs[target_id].detach().float().cpu().item())
            clean_top_id = int(torch.argmax(probs).detach().cpu().item())
            clean_top_probability = float(probs[clean_top_id].detach().float().cpu().item())
            if REQUIRE_CORRECT_CLEAN and clean_top_id != target_id:
                raise TraceSkip('clean_mismatch', f"clean-token mismatch: top={tokenizer.decode([clean_top_id])!r} p={clean_top_probability:.6g}; target={target_first_text!r} p={clean_probability:.6g}")
            noise = make_noise(NUM_NOISE_SAMPLES, len(subject_positions), hidden_size, noise_std, inputs['input_ids'].device, next(model.parameters()).dtype, SEED + int(prompt_idx))
            noise_batch_size = max(1, min(int(NOISE_BATCH_SIZE), int(NUM_NOISE_SAMPLES)))
            corrupt_probabilities = np.zeros((NUM_NOISE_SAMPLES,), dtype=np.float32)
            for batch_start in range(0, NUM_NOISE_SAMPLES, noise_batch_size):
                batch_end = min(NUM_NOISE_SAMPLES, batch_start + noise_batch_size)
                repeated = repeat_inputs(inputs, batch_end - batch_start)
                with temporary_hooks([(emb, corrupt_hook(subject_positions, noise[batch_start:batch_end]))]):
                    corrupt = model(**repeated, use_cache=False)
                corrupt_probabilities[batch_start:batch_end] = token_probability(corrupt.logits, target_id)
                del repeated, corrupt
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        if not np.all(np.isfinite(corrupt_probabilities)) or not np.isfinite(clean_probability):
            raise TraceSkip('nonfinite_output', 'non-finite clean/corrupt probability')
        total_effect = float(clean_probability - np.mean(corrupt_probabilities))
        if total_effect < MIN_TOTAL_EFFECT:
            raise TraceSkip('low_corruption_effect', f'total_effect={total_effect:.6g} < {MIN_TOTAL_EFFECT}')
        corrupt_relative_std = float(np.std(corrupt_probabilities) / max(abs(total_effect), 1e-9))
        windows = trace_mlp_windows_for_fact(inputs, target_id, subject_positions, noise, corrupt_probabilities, clean_probability, subject_last)
        return FactTrace(prompt_idx=int(prompt_idx), prompt=prompt, subject=subject, target_full_text=target, target_first_token_id=target_id, target_first_token_text=target_first_text, target_num_tokens=target_num_tokens, clean_top_token=tokenizer.decode([clean_top_id]), clean_probability=clean_probability, clean_top_probability=clean_top_probability, corrupt_probabilities=corrupt_probabilities, total_effect=total_effect, corrupt_relative_std=corrupt_relative_std, subject_positions=subject_positions, subject_tokens=[decode_position(inputs['input_ids'], pos) for pos in subject_positions], subject_last_position=subject_last, subject_last_token=decode_position(inputs['input_ids'], subject_last), prompt_last_position=prompt_last, prompt_last_token=decode_position(inputs['input_ids'], prompt_last), windows=windows)

    # Collect facts.
    target_valid_facts = int(MODEL_VALID_FACT_LIMITS.get(model_config, NUM_VALID_FACTS))
    max_scan_limit = int(MODEL_MAX_DATASET_EXAMPLES_TO_SCAN.get(model_config, MAX_DATASET_EXAMPLES_TO_SCAN))
    max_trace_seconds = MODEL_TRACE_MAX_SECONDS.get(model_config, TRACE_MAX_SECONDS_PER_MODEL)
    max_trace_seconds = None if max_trace_seconds is None else float(max_trace_seconds)
    trace_started_at = time.time()
    fact_results = []
    rejections = []
    counts = {
        'num_dataset_examples_scanned': 0,
        'num_clean_prediction_matches': 0,
        'num_baseline_reliable_facts': 0,
        'num_rejected_clean_mismatch': 0,
        'num_rejected_low_corruption_effect': 0,
        'num_rejected_nonfinite_output': 0,
        'num_rejected_other': 0,
        'target_valid_facts': target_valid_facts,
        'trace_max_seconds': max_trace_seconds,
        'trace_stopped_by_time_budget': False,
    }
    max_scan = min(max_scan_limit, len(dataset))
    print(f'Tracing {model_config}: collecting {target_valid_facts} valid facts from up to {max_scan} scanned rows. Covariance/ROME start only after layer selection.', flush=True)
    progress = tqdm(range(max_scan), total=max_scan, desc=f'{model_config} facts')
    for prompt_idx in progress:
        if len(fact_results) >= target_valid_facts:
            break
        if max_trace_seconds is not None and (time.time() - trace_started_at) > max_trace_seconds and len(fact_results) >= int(MIN_FACTS_AFTER_TRACE_TIMEOUT):
            counts['trace_stopped_by_time_budget'] = True
            print(f'TRACE TIME BUDGET {model_config}: elapsed={(time.time() - trace_started_at):.1f}s valid={len(fact_results)}; proceeding with collected facts.', flush=True)
            break
        counts['num_dataset_examples_scanned'] += 1
        row = dataset[int(prompt_idx)]
        try:
            prompt, subject, target = row_to_fact(row)
            trace = trace_fact(prompt_idx, prompt, subject, target)
            counts['num_clean_prediction_matches'] += 1
            counts['num_baseline_reliable_facts'] += 1
            fact_results.append(trace)
            full = [w for w in trace.windows if w.is_full_width]
            best = max(full, key=lambda w: w.mean_ie)
            progress.set_postfix(valid=len(fact_results), scanned=counts['num_dataset_examples_scanned'], clean_mismatch=counts['num_rejected_clean_mismatch'])
            print(f"OK {len(fact_results):03d}/{target_valid_facts}: idx={prompt_idx} {subject!r}->{target!r} total={trace.total_effect:.4f} best_center={best.center} IE={best.mean_ie:.4g}", flush=True)
        except TraceSkip as exc:
            try:
                _prompt, subject, target = row_to_fact(row)
            except Exception:
                subject, target = '<unknown>', '<unknown>'
            if exc.reason == 'clean_mismatch':
                counts['num_rejected_clean_mismatch'] += 1
            elif exc.reason == 'low_corruption_effect':
                counts['num_clean_prediction_matches'] += 1
                counts['num_rejected_low_corruption_effect'] += 1
            elif exc.reason == 'nonfinite_output':
                counts['num_rejected_nonfinite_output'] += 1
            else:
                counts['num_rejected_other'] += 1
            rejections.append({'prompt_idx': prompt_idx, 'subject': subject, 'target': target, 'reason': exc.reason, 'detail': exc.detail})
            progress.set_postfix(valid=len(fact_results), scanned=counts['num_dataset_examples_scanned'], clean_mismatch=counts['num_rejected_clean_mismatch'])
            if TRACE_STATUS_EVERY and counts['num_dataset_examples_scanned'] % int(TRACE_STATUS_EVERY) == 0:
                print(f"TRACE {model_config}: scanned={counts['num_dataset_examples_scanned']} valid={len(fact_results)}/{target_valid_facts} clean_mismatch={counts['num_rejected_clean_mismatch']} low_effect={counts['num_rejected_low_corruption_effect']}", flush=True)
    counts['trace_elapsed_seconds'] = float(time.time() - trace_started_at)
    if len(fact_results) < 2:
        raise RuntimeError(f'{model_config}: need at least two baseline-reliable facts')

    summary = pd.DataFrame([{'prompt_idx': t.prompt_idx, 'subject': t.subject, 'target_full_text': t.target_full_text, 'target_first_token_id': t.target_first_token_id, 'target_first_token_text': repr(t.target_first_token_text), 'target_num_tokens': t.target_num_tokens, 'clean_top_token': repr(t.clean_top_token), 'clean_probability': t.clean_probability, 'mean_corrupt_probability': t.mean_corrupt_probability, 'total_effect': t.total_effect, 'corrupt_relative_std': t.corrupt_relative_std, 'subject_positions': t.subject_positions, 'subject_tokens': t.subject_tokens, 'subject_last_position': t.subject_last_position, 'subject_last_token': repr(t.subject_last_token), 'prompt_last_position': t.prompt_last_position, 'prompt_last_token': repr(t.prompt_last_token)} for t in fact_results])
    rejections_df = pd.DataFrame(rejections)

    rng = np.random.default_rng(SEED)
    indices = rng.permutation(len(fact_results))
    discovery_count = max(1, int(round(len(fact_results) * DISCOVERY_FRACTION)))
    discovery_count = min(discovery_count, len(fact_results) - 1)
    discovery_indices = sorted(indices[:discovery_count].tolist())
    confirmation_indices = sorted(indices[discovery_count:].tolist())
    discovery_traces = [fact_results[i] for i in discovery_indices]
    confirmation_traces = [fact_results[i] for i in confirmation_indices]
    split_by_index = {idx: 'discovery' for idx in discovery_indices}
    split_by_index.update({idx: 'confirmation' for idx in confirmation_indices})
    split_assignments = pd.DataFrame([
        {
            'fact_result_index': idx,
            'prompt_idx': int(trace.prompt_idx),
            'subject': trace.subject,
            'split': split_by_index[idx],
        }
        for idx, trace in enumerate(fact_results)
    ])

    def traces_to_matrices(traces):
        centers = np.arange(num_layers)
        ie = np.zeros((len(traces), num_layers), dtype=np.float32)
        norm = np.zeros((len(traces), num_layers), dtype=np.float32)
        restore = np.zeros((len(traces), num_layers, NUM_NOISE_SAMPLES), dtype=np.float32)
        corrupt = np.zeros((len(traces), NUM_NOISE_SAMPLES), dtype=np.float32)
        for fact_i, trace in enumerate(traces):
            corrupt[fact_i] = trace.corrupt_probabilities
            for window in trace.windows:
                ie[fact_i, window.center] = window.mean_ie
                norm[fact_i, window.center] = window.normalized_recovery
                restore[fact_i, window.center, :] = window.restore_probabilities
        return centers, ie, norm, restore, corrupt

    def aggregate_stats(ie_matrix):
        return {
            'mean_ie': ie_matrix.mean(axis=0),
            'std_ie': ie_matrix.std(axis=0),
            'sem_ie': ie_matrix.std(axis=0) / max(np.sqrt(ie_matrix.shape[0]), 1.0),
        }

    centers, discovery_ie, discovery_norm, discovery_restore_probs, discovery_corrupt_probs = traces_to_matrices(discovery_traces)
    discovery_stats = aggregate_stats(discovery_ie)
    discovery_ci_low, discovery_ci_high = bootstrap_ci(discovery_ie, seed=SEED + 100)
    discovery_windows = pd.DataFrame([
        {
            **window_metadata(int(center)),
            'num_facts': len(discovery_traces),
            'discovery_mean_ie': float(discovery_stats['mean_ie'][center]),
            'discovery_std_ie': float(discovery_stats['std_ie'][center]),
            'discovery_sem_ie': float(discovery_stats['sem_ie'][center]),
            'discovery_ci_lower': float(discovery_ci_low[center]),
            'discovery_ci_upper': float(discovery_ci_high[center]),
        }
        for center in centers
    ])
    eligible_discovery = discovery_windows[discovery_windows['window_is_full_width']].copy()
    if eligible_discovery.empty:
        raise RuntimeError(f'{model_config}: no full-width windows; reduce WINDOW_SIZE')
    discovery_row = eligible_discovery.sort_values(
        ['discovery_mean_ie', 'window_center'],
        ascending=[False, True],
    ).iloc[0]
    discovery_center = int(discovery_row['window_center'])
    discovery_window = window_metadata(discovery_center)

    centers, confirmation_ie, confirmation_norm, confirmation_restore_probs, confirmation_corrupt_probs = traces_to_matrices(confirmation_traces)
    confirmation_stats = aggregate_stats(confirmation_ie)
    confirmation_ci_low, confirmation_ci_high = bootstrap_ci(confirmation_ie, seed=SEED + 200)
    confirmation_windows = pd.DataFrame([
        {
            **window_metadata(int(center)),
            'num_facts': len(confirmation_traces),
            'confirmation_mean_ie': float(confirmation_stats['mean_ie'][center]),
            'confirmation_std_ie': float(confirmation_stats['std_ie'][center]),
            'confirmation_sem_ie': float(confirmation_stats['sem_ie'][center]),
            'confirmation_ci_lower': float(confirmation_ci_low[center]),
            'confirmation_ci_upper': float(confirmation_ci_high[center]),
            'preselected_for_confirmation': bool(int(center) == discovery_center),
        }
        for center in centers
    ])
    confirmation_row = confirmation_windows[
        confirmation_windows['window_center'] == discovery_center
    ].iloc[0]
    enough_confirmation_facts = len(confirmation_traces) >= int(MIN_CONFIRMATION_FACTS)
    confirmation_passed = bool(
        enough_confirmation_facts
        and np.isfinite(float(confirmation_row['confirmation_ci_lower']))
        and float(confirmation_row['confirmation_ci_lower']) > 0
    )
    if not enough_confirmation_facts:
        selection_failure_reason = 'insufficient_confirmation_facts'
    elif not confirmation_passed:
        selection_failure_reason = 'confirmation_ci_not_positive'
    else:
        selection_failure_reason = None

    selected_trace_center = discovery_center if confirmation_passed else None
    selected_window_layers = [int(layer) for layer in discovery_window['window_layers']]
    selection_diagnostics = {
        'selection_method': 'discovery_argmax_then_held_out_confirmation',
        'eligible_window_rule': 'full_width_only',
        'tie_break_rule': 'lower_center',
        'minimum_confirmation_facts': int(MIN_CONFIRMATION_FACTS),
        'confirmation_passed': confirmation_passed,
        'selection_failure_reason': selection_failure_reason,
        'discovery_mean_ie': float(discovery_row['discovery_mean_ie']),
        'confirmation_mean_ie': float(confirmation_row['confirmation_mean_ie']),
        'confirmation_ci_lower': float(confirmation_row['confirmation_ci_lower']),
        'confirmation_ci_upper': float(confirmation_row['confirmation_ci_upper']),
    }

    trace_plot_path = out_dir / 'early_site_trace_v2.png'
    final_selection = {
        'model_config': model_config,
        'model_name': model_name,
        'trace_plot_path': str(trace_plot_path),
        'selection_method': 'discovery_argmax_then_held_out_confirmation',
        'graph_selected_benchmark_layer': selected_trace_center,
        'graph_selected_layer_source': (
            'discovery_argmax_held_out_positive_ci' if confirmation_passed else None
        ),
        'held_out_confirmed_window': confirmation_passed,
        'selection_failure_reason': selection_failure_reason,
        'strict_selection_failure_reason': selection_failure_reason,
        'selected_trace_center': selected_trace_center,
        'discovery_trace_center': discovery_center,
        'trace_window_start': int(discovery_window['window_start']),
        'trace_window_end': int(discovery_window['window_end']),
        'trace_window_layers': selected_window_layers,
        'config_layer_used_for_selection': False,
        'config_reference_layer': None if config_reference_layer is None else int(config_reference_layer),
        'config_reference_layer_is_graph_only': True,
        'adapter_validate_all_layers': ADAPTER_VALIDATE_ALL_LAYERS,
        'trust_remote_code_used': model_trust_remote_code,
        'selection_diagnostics': selection_diagnostics,
        'best_validated_rome_edit_layer': None,
        'num_fact_results': len(fact_results),
        'num_discovery_facts': len(discovery_traces),
        'num_confirmation_facts': len(confirmation_traces),
    }
    print(json.dumps(final_selection, indent=2))
    if not confirmation_passed:
        print(
            f'No held-out-confirmed window selected: {selection_failure_reason}. '
            f'Discovery center={discovery_center}.',
            flush=True,
        )

    full_mask = confirmation_windows['window_is_full_width'].to_numpy(dtype=bool)
    partial_mask = ~full_mask
    fig, ax = plt.subplots(figsize=(15.5, 6.5))
    ax.bar(
        centers[partial_mask],
        confirmation_stats['mean_ie'][partial_mask],
        color='lightgray',
        label='partial boundary windows',
    )
    ax.bar(
        centers[full_mask],
        confirmation_stats['mean_ie'][full_mask],
        color='steelblue',
        alpha=0.55,
        label='confirmation mean IE',
    )
    ax.fill_between(
        centers,
        confirmation_ci_low,
        confirmation_ci_high,
        color='tab:blue',
        alpha=0.15,
        label=f'{int(CONFIDENCE_LEVEL * 100)}% confirmation CI',
    )
    ax.plot(
        centers,
        discovery_stats['mean_ie'],
        color='black',
        linewidth=1.8,
        linestyle='--',
        label='discovery mean IE',
    )
    ax.axhline(0.0, color='black', linewidth=0.8)
    selection_color = 'tab:green' if confirmation_passed else 'darkorange'
    selection_label = 'held-out confirmed' if confirmation_passed else 'not held-out confirmed'
    ax.axvline(
        discovery_center,
        color=selection_color,
        linewidth=2.8,
        label=f'discovery center {discovery_center} ({selection_label})',
    )
    if config_reference_layer is not None:
        ax.axvline(
            int(config_reference_layer),
            color='tab:purple',
            linestyle='-.',
            linewidth=2.5,
            label=f'config reference layer {config_reference_layer} (graph only)',
        )
    ax.set_title(f'causal-tracing-auto-v2: {model_config}')
    ax.set_xlabel('MLP window center')
    ax.set_ylabel('Mean paired indirect effect across facts')
    ax.legend(fontsize=8, loc='upper left')
    plt.tight_layout()
    if SAVE:
        out_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(trace_plot_path, dpi=180)
    plt.show()

    if SAVE:
        out_dir.mkdir(parents=True, exist_ok=True)
        summary.to_csv(out_dir / 'summary_facts.csv', index=False)
        rejections_df.to_csv(out_dir / 'rejections.csv', index=False)
        split_assignments.to_csv(out_dir / 'split_assignments.csv', index=False)
        discovery_windows.to_csv(out_dir / 'discovery_windows.csv', index=False)
        confirmation_windows.to_csv(out_dir / 'confirmation_windows.csv', index=False)
        (out_dir / 'final_selection.json').write_text(json.dumps(json_safe(final_selection), indent=2))
        (out_dir / 'selection_diagnostics.json').write_text(json.dumps(json_safe(selection_diagnostics), indent=2))
        (out_dir / 'config.json').write_text(json.dumps(json_safe({
            'model_config': model_config,
            'model_preset': preset,
            'trust_remote_code_used': model_trust_remote_code,
            'config_reference_layer': config_reference_layer,
            'config_reference_layer_is_graph_only': True,
            'adapter_validate_all_layers': ADAPTER_VALIDATE_ALL_LAYERS,
            'num_valid_facts_requested': target_valid_facts,
            'num_valid_facts_global_default': NUM_VALID_FACTS,
            'num_noise_samples': NUM_NOISE_SAMPLES,
            'noise_batch_size': NOISE_BATCH_SIZE,
            'window_mode': WINDOW_MODE,
            'window_size': primary_window_size,
            'discovery_fraction': DISCOVERY_FRACTION,
            'minimum_confirmation_facts': MIN_CONFIRMATION_FACTS,
            'bootstrap_samples': BOOTSTRAP_SAMPLES,
            'confidence_level': CONFIDENCE_LEVEL,
            'seed': SEED,
            'counts': counts,
            'mlp_output_modules': module_map.to_dict(orient='records'),
        }), indent=2))
        with (out_dir / 'fact_results.jsonl').open('w') as f:
            for trace in fact_results:
                f.write(json.dumps(json_safe({
                    'prompt_idx': trace.prompt_idx,
                    'prompt': trace.prompt,
                    'subject': trace.subject,
                    'target_full_text': trace.target_full_text,
                    'target_first_token_id': trace.target_first_token_id,
                    'target_first_token_text': trace.target_first_token_text,
                    'target_num_tokens': trace.target_num_tokens,
                    'clean_probability': trace.clean_probability,
                    'mean_corrupt_probability': trace.mean_corrupt_probability,
                    'total_effect': trace.total_effect,
                    'corrupt_relative_std': trace.corrupt_relative_std,
                    'windows': [
                        {
                            'window_center': w.center,
                            'window_start': w.start,
                            'window_end': w.end,
                            'window_size_actual': len(w.layers),
                            'window_layers': w.layers,
                            'window_is_full_width': w.is_full_width,
                            'mean_ie': w.mean_ie,
                            'median_ie_diagnostic': w.median_ie,
                            'mean_normalized_recovery_diagnostic': w.normalized_recovery,
                        }
                        for w in trace.windows
                    ],
                })) + '\n')
        np.savez_compressed(
            out_dir / 'raw_window_probabilities.npz',
            discovery_ie=discovery_ie,
            confirmation_ie=confirmation_ie,
            discovery_restore_probabilities=discovery_restore_probs,
            confirmation_restore_probabilities=confirmation_restore_probs,
            discovery_corrupt_probabilities=discovery_corrupt_probs,
            confirmation_corrupt_probabilities=confirmation_corrupt_probs,
            discovery_norm_diagnostic=discovery_norm,
            confirmation_norm_diagnostic=confirmation_norm,
            discovery_ci_lower=discovery_ci_low,
            discovery_ci_upper=discovery_ci_high,
            confirmation_ci_lower=confirmation_ci_low,
            confirmation_ci_upper=confirmation_ci_high,
        )
        print(f'Wrote outputs to {out_dir}')

    result = {**final_selection, **counts, 'out_dir': str(out_dir)}
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


## 8. Run Causal Tracing

This stage runs paired subject-last MLP-window tracing. Discovery chooses the highest-mean full-width window once; held-out confirmation only tests that window. It saves the exact split, per-model CSV/JSON/NPZ outputs, and a PNG graph.


In [ ]:
batch_results = []
trace_by_model = {}

if RUN_CAUSAL_TRACE:
    for model_config in MODEL_CONFIGS:
        print(f'\n##### CAUSAL TRACE MODEL: {model_config} #####', flush=True)
        trace = run_one_model(model_config, dataset)
        batch_results.append(trace)
        trace_by_model[model_config] = trace
else:
    print('RUN_CAUSAL_TRACE=False; skipping causal tracing.')

batch_summary = pd.DataFrame(batch_results)
if len(batch_summary):
    display(batch_summary)

if SAVE:
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    batch_summary_path = OUT_ROOT / f'causal_trace_batch_summary_{RUN_TIMESTAMP}.csv'
    batch_summary.to_csv(batch_summary_path, index=False)
    print(f'Wrote causal trace batch summary to {batch_summary_path}')


## 9. Trace Layer Selection Helpers

These helpers pass only a held-out-confirmed trace center to the optional covariance and ROME stages. The discovery center remains diagnostic when confirmation fails. If the confirmed center equals the config/reference layer, the separate covariance stage skips it because there is no new layer to prepare.


In [ ]:
def trace_layer_for_downstream(model_config, trace):
    if not trace:
        return None, 'no_causal_trace_result'
    layer = trace.get('graph_selected_benchmark_layer')
    source = trace.get('graph_selected_layer_source') or 'discovery_argmax_held_out_positive_ci'
    if layer is None:
        return None, trace.get('strict_selection_failure_reason') or 'no_held_out_confirmed_trace_center'
    selected_layer = int(layer)
    config_layer = trace.get('config_reference_layer')
    if config_layer is not None and selected_layer == int(config_layer):
        return None, f'graph_selected_layer_matches_existing_config:{selected_layer}'
    return selected_layer, source

def config_reference_layer_for_model(model_config, trace=None):
    if trace and trace.get('config_reference_layer') is not None:
        return int(trace['config_reference_layer'])
    value = CONFIG_REFERENCE_LAYERS.get(model_config)
    return None if value is None else int(value)

def should_retry_config_layer_after_failed_rome(model_config, trace, attempted_layer, result):
    if not ROME_RETRY_CONFIG_LAYER_ON_ZERO_EVALUATED:
        return False, None, 'config_retry_disabled'
    if result is None:
        return False, None, 'no_rome_result'
    summary = result.get('summary', {})
    if int(summary.get('n_evaluated', 0) or 0) > 0:
        return False, None, 'primary_layer_evaluated_cases'
    config_layer = config_reference_layer_for_model(model_config, trace)
    if config_layer is None:
        return False, None, 'no_config_reference_layer'
    if attempted_layer is not None and int(attempted_layer) == int(config_layer):
        return False, None, 'attempted_layer_is_config_reference'
    return True, int(config_layer), 'primary_layer_zero_evaluated_retry_config_reference'

def skipped_result(model_config, stage, reason):
    return {'model_config': model_config, 'stage': stage, 'skipped': True, 'skip_reason': reason}


## 10. Self-Contained ROME Configuration

Everything below is only needed for the optional covariance/ROME cells. The causal tracing stage above does not depend on these cells.


In [ ]:

import logging
import re
from enum import Enum
from typing import Any, Sequence

LOGGER = logging.getLogger('causal_tracing_auto')
if not LOGGER.handlers:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

ROME_BASE_CONFIG = dict(
    lr=0.5,
    kl_factor=0.0625,
    weight_decay=0.5,
    epochs=20,
    k_N=50,
    v_N=50,
    prefix_range=[2, 10],
    optimize_v_clear_cache_every=0,
    prefix_mode='self',
    prefix_source=None,
    prefix_cache_path=None,
    prefix_cache_size=256,
    prefix_enforce_latin=False,
    prefix_min_words=0,
    second_moment_dir=str(SECOND_MOMENT_DIR),
    second_moment_batch_size_mode=SECOND_MOMENT_BATCH_SIZE_MODE,
    second_moment_batch_size=SECOND_MOMENT_BATCH_SIZE,
    second_moment_target_samples=SECOND_MOMENT_TARGET_SAMPLES,
    second_moment_max_length=SECOND_MOMENT_MAX_LENGTH,
    second_moment_min_text_length=SECOND_MOMENT_MIN_TEXT_LENGTH,
    second_moment_clear_cache_every=SECOND_MOMENT_CLEAR_CACHE_EVERY,
)

ROME_MODEL_CONFIGS = {
    'gpt2-medium': dict(name='gpt2-medium', layer=8, layer_name_template='transformer.h.{}.mlp.c_proj', fc_layer_name_template=None, corrupt_layer_name_template='transformer.wte', second_moment_path='./data/second_moment_stats/gpt2-medium_8_SM_Method.WIKIPEDIA_100000.pt', dtype='bf16'),
    'gpt2-large': dict(name='gpt2-large', layer=12, layer_name_template='transformer.h.{}.mlp.c_proj', fc_layer_name_template='transformer.h.{}.mlp.c_fc', corrupt_layer_name_template='transformer.wte', second_moment_path='./data/second_moment_stats/gpt2-large_12_SM_Method.WIKIPEDIA_100000.pt', dtype='bf16'),
    'gpt2-xl': dict(name='gpt2-xl', layer=17, layer_name_template='transformer.h.{}.mlp.c_proj', fc_layer_name_template='transformer.h.{}.mlp.c_fc', corrupt_layer_name_template='transformer.wte', second_moment_path='./data/second_moment_stats/gpt2-xl_17_SM_Method.WIKIPEDIA_100000.pt', dtype='bf16'),
    'gpt-j-6b': dict(name='EleutherAI/gpt-j-6B', layer=5, layer_name_template='transformer.h.{}.mlp.fc_out', fc_layer_name_template='transformer.h.{}.mlp.fc_in', corrupt_layer_name_template='transformer.wte', second_moment_path='./data/second_moment_stats/EleutherAI_gpt-j-6B_5_SM_Method.WIKIPEDIA_100000.pt', dtype='bf16'),
    'falcon-7b': dict(name='tiiuae/falcon-7b', layer=5, layer_name_template='transformer.h.{}.mlp.dense_4h_to_h', fc_layer_name_template='transformer.h.{}.mlp.dense_h_to_4h', corrupt_layer_name_template='transformer.word_embeddings', second_moment_path='./data/second_moment_stats/tiiuae_falcon-7b_5_SM_Method.WIKIPEDIA_100000.pt', epochs=10, dtype='bf16', trust_remote_code=False),
    'opt-6.7b': dict(name='facebook/opt-6.7b', layer=15, layer_name_template='model.decoder.layers.{}.fc2', fc_layer_name_template='model.decoder.layers.{}.fc1', corrupt_layer_name_template='model.decoder.embed_tokens', second_moment_path='./data/second_moment_stats/facebook_opt-6.7b_15_SM_Method.WIKIPEDIA_100000.pt', epochs=10, dtype='bf16'),
    'llama2-7b': dict(name='NousResearch/Llama-2-7b-hf', layer=6, layer_name_template='model.layers.{}.mlp.down_proj', fc_layer_name_template='model.layers.{}.mlp.up_proj', corrupt_layer_name_template='model.embed_tokens', second_moment_path='./data/second_moment_stats/NousResearch_Llama-2-7b-hf_6_SM_Method.WIKIPEDIA_100000.pt', dtype='bf16'),
    'mistral-7b-v0.1': dict(name='mistralai/Mistral-7B-v0.1', layer=5, layer_name_template='model.layers.{}.mlp.down_proj', fc_layer_name_template='model.layers.{}.mlp.up_proj', corrupt_layer_name_template='model.embed_tokens', second_moment_path='./data/second_moment_stats/mistralai_Mistral-7B-v0.1_5_SM_Method.WIKIPEDIA_100000.pt', dtype='bf16'),
    'mistral-7b-v0.3': dict(name='mistralai/Mistral-7B-v0.3', layer=17, layer_name_template='model.layers.{}.mlp.down_proj', fc_layer_name_template='model.layers.{}.mlp.up_proj', corrupt_layer_name_template='model.embed_tokens', second_moment_path='./data/second_moment_stats/mistralai_Mistral-7B-v0.3_17_SM_Method.WIKIPEDIA_100000.pt', dtype='bf16', prefix_enforce_latin=True, prefix_min_words=2),
    'deepseek-7b-base': dict(name='deepseek-ai/deepseek-llm-7b-base', layer=20, layer_name_template='model.layers.{}.mlp.down_proj', fc_layer_name_template='model.layers.{}.mlp.up_proj', corrupt_layer_name_template='model.embed_tokens', second_moment_path='./data/second_moment_stats/deepseek-ai_deepseek-llm-7b-base_20_SM_Method.WIKIPEDIA_100000.pt', epochs=25, dtype='bf16', prefix_mode='external', prefix_source='mistral-7b-v0.3', prefix_cache_path='./prefix_cache/deepseek-7b-base__mistral-7b-v0.3.json', prefix_cache_size=256, prefix_enforce_latin=True, prefix_min_words=0),
    'deepseek-r1-llama3-8b': dict(name='deepseek-ai/DeepSeek-R1-Distill-Llama-8B', layer=0, layer_name_template='model.layers.{}.mlp.down_proj', fc_layer_name_template=None, corrupt_layer_name_template='model.embed_tokens', second_moment_path=None, dtype='bf16'),
    'qwen2.5-1.5b': dict(name='Qwen/Qwen2.5-Math-1.5B', layer=7, layer_name_template='model.layers.{}.mlp.down_proj', fc_layer_name_template=None, corrupt_layer_name_template='model.embed_tokens', second_moment_path=None, dtype='bf16'),
    'qwen3-0.6b': dict(name='Qwen/Qwen3-0.6B', layer=5, layer_name_template='model.layers.{}.mlp.down_proj', fc_layer_name_template=None, corrupt_layer_name_template='model.embed_tokens', second_moment_path=None, dtype='bf16'),
    'qwen3-1.7b': dict(name='Qwen/Qwen3-1.7B', layer=9, layer_name_template='model.layers.{}.mlp.down_proj', fc_layer_name_template=None, corrupt_layer_name_template='model.embed_tokens', second_moment_path=None, dtype='bf16'),
    'qwen3-4b': dict(name='Qwen/Qwen3-4B', layer=6, layer_name_template='model.layers.{}.mlp.down_proj', fc_layer_name_template='model.layers.{}.mlp.up_proj', corrupt_layer_name_template='model.embed_tokens', second_moment_path='./data/second_moment_stats/Qwen_Qwen3-4B_6_SM_Method.WIKIPEDIA_100000.pt', dtype='bf16', prefix_source='mistral-7b-v0.3', prefix_cache_path='./prefix_cache/qwen3-4b__mistral-7b-v0.3.json'),
    'qwen3-8b': dict(name='Qwen/Qwen3-8B', layer=7, layer_name_template='model.layers.{}.mlp.down_proj', fc_layer_name_template='model.layers.{}.mlp.up_proj', corrupt_layer_name_template='model.embed_tokens', second_moment_path='./data/second_moment_stats/Qwen_Qwen3-8B_7_SM_Method.WIKIPEDIA_100000.pt', dtype='bf16', prefix_mode='external', prefix_source='mistral-7b-v0.3', prefix_cache_path='./prefix_cache/qwen3-8b__mistral-7b-v0.3.json', prefix_cache_size=256),
    'granite4-micro': dict(name='ibm-granite/granite-4.0-micro', layer=35, layer_name_template='model.layers.{}.shared_mlp.output_linear', fc_layer_name_template=None, corrupt_layer_name_template='model.embed_tokens', second_moment_path='./data/second_moment_stats/ibm-granite_granite-4.0-micro_35_SM_Method.WIKIPEDIA_100000.pt', dtype='bf16'),
    'qwen3-guard-0.6b': dict(name='Qwen/Qwen3Guard-Gen-0.6B', layer=5, layer_name_template='model.layers.{}.mlp.down_proj', fc_layer_name_template=None, corrupt_layer_name_template='model.embed_tokens', second_moment_path=None, dtype='bf16'),
}
for alias in ['qwen3-8b-prefixtest-external-long', 'qwen3-8b-prefixtest-external-medium', 'qwen3-8b-prefixtest-external-short', 'qwen3-8b-prefixtest-self-long', 'qwen3-8b-prefixtest-self-medium', 'qwen3-8b-prefixtest-self-short', 'qwen3-8b-prefixtest-template-long', 'qwen3-8b-prefixtest-template-medium', 'qwen3-8b-prefixtest-template-short']:
    ROME_MODEL_CONFIGS[alias] = dict(ROME_MODEL_CONFIGS['qwen3-8b'])
for key, value in list(ROME_MODEL_CONFIGS.items()):
    merged = dict(ROME_BASE_CONFIG)
    merged.update(value)
    ROME_MODEL_CONFIGS[key] = merged
missing = [name for name in MODEL_CONFIGS if name not in ROME_MODEL_CONFIGS]
if missing:
    raise ValueError(f'Missing ROME model configs: {missing}')
print('ROME configs OK')


## 11. Self-Contained ROME Primitives

ROME helper code copied into the notebook so it can run outside the repo, including Colab-style environments.


In [ ]:

class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError as exc:
            raise AttributeError(key) from exc
    def __setattr__(self, key, value):
        self[key] = value

def to_attr_dict(value):
    if isinstance(value, dict):
        return AttrDict({k: to_attr_dict(v) for k, v in value.items()})
    if isinstance(value, list):
        return [to_attr_dict(v) for v in value]
    return value

class NotebookDeviceManager:
    def __init__(self, preferred_device):
        self.preferred_device = preferred_device
    def safe_to_device(self, data, device=None):
        return data.to(device or self.preferred_device)
    def clear_cache(self):
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    def register_object(self, _obj):
        return None

_STATIC_PREFIXES = ['{}', 'As a fact, {}', 'In one sentence, {}', 'Historically, {}', 'In summary, {}', 'It is known that {}', 'For context, {}', 'In plain terms, {}', 'To clarify, {}', 'A key point: {}', 'By definition, {}']
_LATIN_ONLY_RE = re.compile(r'^[\x20-\x7E\u00C0-\u024F\u1E00-\u1EFF]*$')

def _dedupe_templates(templates):
    out, seen = [], set()
    for template in templates:
        template = str(template).strip()
        if template and template not in seen:
            out.append(template); seen.add(template)
    return out

def resolve_prefix_range(handler, prefix_range=None):
    raw = prefix_range if prefix_range is not None else getattr(handler.cfg.model, 'prefix_range', None)
    if raw is None:
        raw = getattr(handler.cfg.generation, 'prefix_range', None)
    lo = max(1, int(raw[0])); hi = max(lo, int(raw[1]))
    return lo, hi

def generate_prefixes(handler, N, prefix_range=None, additional_prompts=None):
    count = max(1, int(N)); mode = str(getattr(handler.cfg.model, 'prefix_mode', 'self') or 'self')
    pool = []
    if mode == 'external' and getattr(handler.cfg.model, 'prefix_cache_path', None):
        path = Path(handler.cfg.model.prefix_cache_path).expanduser()
        if path.exists():
            payload = json.loads(path.read_text())
            pool = payload.get('templates', payload) if isinstance(payload, dict) else payload
    if not pool:
        pool = list(_STATIC_PREFIXES)
    pool = _dedupe_templates(pool)
    templates = []
    while len(templates) < count:
        chunk = list(pool); random.shuffle(chunk); templates.extend(chunk)
    templates = templates[:count]
    if bool(getattr(handler.cfg.model, 'prefix_enforce_latin', False)) or int(getattr(handler.cfg.model, 'prefix_min_words', 0) or 0) > 0:
        filtered = []
        for template in templates:
            body = template.replace('{}', '').strip()
            if bool(getattr(handler.cfg.model, 'prefix_enforce_latin', False)) and not _LATIN_ONLY_RE.match(body):
                continue
            if int(getattr(handler.cfg.model, 'prefix_min_words', 0) or 0) > 0 and len(body.split()) < int(handler.cfg.model.prefix_min_words):
                continue
            filtered.append(template)
        templates = filtered or _STATIC_PREFIXES[:count]
    return templates + list(additional_prompts or [])

class NotebookRomeHandler:
    def __init__(self, model_config, layer_override=None):
        self.model_config = model_config
        raw = dict(ROME_MODEL_CONFIGS[model_config])
        preset = dict(MODEL_PRESETS.get(model_config, {}))
        trust_remote_code = bool(raw.get('trust_remote_code', preset.get('trust_remote_code', TRUST_REMOTE_CODE)))
        dtype_picker = {'auto': 'auto', 'bf16': torch.bfloat16, 'f16': torch.float16, 'f32': torch.float32}
        dtype = dtype_picker.get(DTYPE, dtype_picker.get(raw.get('dtype', 'auto'), 'auto'))
        kwargs = {'trust_remote_code': trust_remote_code}
        if dtype != 'auto':
            kwargs['torch_dtype'] = dtype
        if os.environ.get('HF_TOKEN'):
            kwargs['token'] = os.environ['HF_TOKEN']
        if USE_DEVICE_MAP_AUTO:
            kwargs['device_map'] = 'auto'
        if LOAD_IN_4BIT:
            kwargs['load_in_4bit'] = True; kwargs['device_map'] = 'auto'
        print(f'Loading ROME model {model_config}: {raw["name"]} trust_remote_code={trust_remote_code}')
        self.model = AutoModelForCausalLM.from_pretrained(raw['name'], **kwargs)
        if not USE_DEVICE_MAP_AUTO and not LOAD_IN_4BIT:
            self.model = self.model.to(DEVICE)
        tok_kwargs = {'trust_remote_code': trust_remote_code}
        if os.environ.get('HF_TOKEN'):
            tok_kwargs['token'] = os.environ['HF_TOKEN']
        self.tokenizer = AutoTokenizer.from_pretrained(raw['name'], **tok_kwargs)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token if self.tokenizer.eos_token is not None else self.tokenizer.pad_token
            if self.tokenizer.pad_token_id is None and self.tokenizer.eos_token_id is not None:
                self.tokenizer.pad_token_id = self.tokenizer.eos_token_id
        self.model.eval()
        self.dtype = next(self.model.parameters()).dtype
        self.device = next(self.model.parameters()).device
        self.device_manager = NotebookDeviceManager(self.device)
        self.cfg = to_attr_dict({'model': raw, 'generation': {'k_N': raw['k_N'], 'v_N': raw['v_N'], 'prefix_range': raw['prefix_range']}, 'runtime': {'second_moment_allow_autocompute': SECOND_MOMENT_ALLOW_AUTOCOMPUTE}})
        self._layer_name_template = raw['layer_name_template']; self._corrupt_layer_name_template = raw['corrupt_layer_name_template']
        self.configured_layer = int(raw['layer'])
        self._layer = int(self.configured_layer if layer_override is None else layer_override)
        raw['layer'] = self._layer
        self.layer_override = None if layer_override is None else int(layer_override)
        self.config_layer_used_for_selection = False
        self.num_of_layers = int(getattr(self.model.config, 'num_hidden_layers', MODEL_PRESETS.get(model_config, {}).get('layers', 0) or 0))
        module = self._get_module(self._layer_name_template.format(self._layer))
        self.emb_shape = min(module.weight.shape); self.hidden_dim = max(module.weight.shape)
        self.second_moment_dir = str(SECOND_MOMENT_DIR)
        configured_second_moment_path = raw.get('second_moment_path')
        self.second_moment_path = None if (self.layer_override is not None and self._layer != self.configured_layer) else configured_second_moment_path
        self.last_second_moment_paths = {}
        self._second_moment_cache = None
        self.epochs = int(raw.get('epochs', 20)); self.lr = float(raw.get('lr', 0.5)); self.kl_factor = float(raw.get('kl_factor', 0.0625)); self.weight_decay = float(raw.get('weight_decay', 0.5))
        self._hooks = []
    def _get_module(self, module_name):
        for name, module in self.model.named_modules():
            if name == module_name:
                return module
        raise KeyError(f'{module_name} not found')
    def get_module_device(self, module_name=None):
        module = self._get_module(module_name or self._layer_name_template.format(self._layer))
        try:
            return next(module.parameters()).device
        except StopIteration:
            return self.device
    def tokenize_prompt(self, prompt):
        return self.tokenizer(prompt, padding=True, return_tensors='pt').to(next(self.model.parameters()).device)
    def set_k_hook(self, fn):
        self._hooks.append(self._get_module(self._layer_name_template.format(self._layer)).register_forward_pre_hook(fn))
    def set_delta_hook(self, fn):
        self._hooks.append(self._get_module(self._layer_name_template.format(self._layer)).register_forward_hook(fn))
    def remove_hooks(self):
        for handle in self._hooks:
            try: handle.remove()
            except Exception: pass
        self._hooks = []
        if torch.cuda.is_available(): torch.cuda.empty_cache()

def _reshape_hidden_states(hidden_states, batch_size, seq_len):
    if hidden_states.dim() == 3:
        return hidden_states, False
    if hidden_states.dim() == 2:
        expected = int(batch_size) * int(seq_len)
        if int(hidden_states.size(0)) != expected:
            raise RuntimeError(f'Unexpected flattened activation rows={hidden_states.size(0)} expected={expected}')
        return hidden_states.view(batch_size, seq_len, hidden_states.size(-1)), True
    raise RuntimeError(f'Unsupported activation rank {hidden_states.dim()}')

def resolve_rome_sample_count(handler, key):
    value = getattr(handler.cfg.model, key, None)
    if value is None: value = getattr(handler.cfg.generation, key, None)
    if value is None: raise ValueError(f'Missing ROME sample count: {key}')
    return max(1, int(value))

def _strip_bos(handler, token_ids):
    bos_id = getattr(handler.tokenizer, 'bos_token_id', None)
    if bos_id is not None and isinstance(token_ids, torch.Tensor):
        if token_ids.dim() == 2 and token_ids.size(1) > 1 and int(token_ids[0, 0]) == int(bos_id): return token_ids[:, 1:]
        if token_ids.dim() == 1 and token_ids.size(0) > 1 and int(token_ids[0]) == int(bos_id): return token_ids[1:]
    return token_ids

def get_subject_position(handler, prompt, subject):
    ids_prompt = handler.tokenize_prompt(prompt)['input_ids']
    ids_subject = _strip_bos(handler, handler.tokenize_prompt(subject)['input_ids'])
    for candidate in [ids_subject, _strip_bos(handler, handler.tokenize_prompt(f' {subject}')['input_ids'])]:
        windows = ids_prompt.unfold(1, candidate.size(1), 1)
        matches = (windows == candidate).all(dim=2)
        pos = list(set(matches.nonzero(as_tuple=True)[1].tolist()))
        if pos:
            return int(pos[0] + candidate.size(1) - 1)
    char_start = prompt.find(subject)
    if char_start != -1:
        char_end = char_start + len(subject)
        try:
            raw = handler.tokenizer(prompt, return_offsets_mapping=True, return_tensors='pt')
            positions = [i for i, (s, e) in enumerate(raw['offset_mapping'][0].tolist()) if e > s and e > char_start and s < char_end]
            if positions: return int(positions[-1])
        except Exception:
            pass
    return -1

def get_subject_index(handler, prompts, fact_tuple, subject_understanding_template):
    new_target_ids = _strip_bos(handler, handler.tokenize_prompt(fact_tuple[2])['input_ids'][0])
    idx = prompts.attention_mask[torch.arange(prompts.input_ids.shape[0], device=prompts.attention_mask.device)].sum(dim=1)
    fact_prompt = handler.tokenize_prompt(fact_tuple[0].format(fact_tuple[1]))
    pos = get_subject_position(handler, fact_tuple[0].format(fact_tuple[1]), fact_tuple[1])
    if pos == -1: return None
    idx[: prompts.input_ids.shape[0] - 1] -= (len(fact_prompt['input_ids'][0]) - pos) + len(new_target_ids) - 1
    u_prompt = handler.tokenize_prompt(subject_understanding_template.format(fact_tuple[1]))
    pos = get_subject_position(handler, subject_understanding_template.format(fact_tuple[1]), fact_tuple[1])
    if pos == -1: return None
    idx[-1] -= len(u_prompt['input_ids'][0]) - pos
    return idx.long().cpu()

def gather_k(handler, fact_tuple, N=50, prefix_range=None, additional_prompts=None):
    templates = [t.format(fact_tuple[1]) for t in generate_prefixes(handler, N, prefix_range, additional_prompts)]
    prompts = handler.tokenize_prompt(templates)
    prompt_count, seq_len = int(prompts.input_ids.shape[0]), int(prompts.input_ids.shape[1])
    token_index = (prompts.attention_mask.detach().cpu().sum(dim=1) - 1).long()
    k = None
    def hook(_module, input):
        nonlocal k
        hidden, _ = _reshape_hidden_states(input[0], prompt_count, seq_len)
        rows = torch.arange(prompt_count, device=hidden.device)
        k = hidden[rows, token_index.to(hidden.device), :].mean(dim=0)
        return input
    handler.set_k_hook(hook)
    with torch.no_grad(): handler.model(**prompts, use_cache=False)
    handler.remove_hooks()
    return k.to(handler.get_module_device(handler._layer_name_template.format(handler._layer)))

def optimize_v(handler, fact_tuple, N_prompts, N_optim_steps, subject_understanding_template='{} is a', prefix_range=None, verbose=False):
    v_init = None; dkl_orig = None
    new_target_ids = _strip_bos(handler, handler.tokenize_prompt(fact_tuple[2])['input_ids'][0])
    main_count = max(1, int(N_prompts))
    templates = generate_prefixes(handler, main_count, resolve_prefix_range(handler, prefix_range), [subject_understanding_template])
    templates = [t.format(fact_tuple[0].format(fact_tuple[1])) for t in templates]
    if new_target_ids.size(0) > 1:
        templates = [t + handler.tokenizer.decode(new_target_ids[:-1]) for t in templates]
    prompts = handler.tokenize_prompt(templates)
    prompt_count, seq_len = int(prompts.input_ids.shape[0]), int(prompts.input_ids.shape[1])
    last_subject_index = get_subject_index(handler, prompts, fact_tuple, subject_understanding_template)
    if last_subject_index is None: return None
    subject_positions = [int(x) for x in last_subject_index.tolist()]
    layer_device = handler.get_module_device(handler._layer_name_template.format(handler._layer))
    delta = torch.zeros((handler.emb_shape), dtype=handler.dtype, device=layer_device).requires_grad_(True)
    opt = torch.optim.Adam([delta], lr=handler.lr)
    residual_mult = float(getattr(handler.model.config, 'residual_multiplier', 1.0)); delta_scale = (1.0 / residual_mult) if 0 < residual_mult < 1.0 else 1.0
    def delta_hook(_module, _input, output):
        nonlocal v_init
        raw = output[0] if isinstance(output, tuple) else output
        hidden, was_flat = _reshape_hidden_states(raw, prompt_count, seq_len)
        patched = hidden.clone()
        if v_init is None: v_init = hidden[0, subject_positions[0]].detach().clone()
        for row, pos in enumerate(subject_positions):
            patched[row, pos, :] = patched[row, pos, :] + (delta * delta_scale).to(device=raw.device, dtype=raw.dtype)
        restored = patched.reshape_as(raw) if was_flat else patched
        if isinstance(output, tuple):
            values = list(output); values[0] = restored; return tuple(values)
        return restored
    target_len = int(new_target_ids.size(0))
    main_idx_cpu = torch.arange(main_count, dtype=torch.long)
    positions_cpu = (prompts.attention_mask[:main_count].detach().cpu().sum(dim=1).unsqueeze(1) - target_len + torch.arange(target_len).unsqueeze(0)).long()
    ids_cpu = new_target_ids.detach().cpu().long().unsqueeze(0).repeat(main_count, 1)
    dkl_idx_cpu = torch.arange(main_count, prompts.input_ids.shape[0], dtype=torch.long)
    dkl_pos_cpu = (prompts.attention_mask.detach().cpu()[dkl_idx_cpu].sum(dim=1) - 1).long()
    for step in range(int(N_optim_steps)):
        opt.zero_grad(); handler.set_delta_hook(delta_hook); outputs = handler.model(**prompts, use_cache=False); handler.remove_hooks()
        device = outputs.logits.device
        all_log_probs = torch.log_softmax(outputs.logits, dim=2)
        log_probs = all_log_probs[main_idx_cpu.to(device).unsqueeze(1), positions_cpu.to(device), ids_cpu.to(device)]
        dkl_logits = outputs.logits[dkl_idx_cpu.to(device), dkl_pos_cpu.to(device), :]
        dkl_log_probs = torch.nn.functional.log_softmax(dkl_logits, dim=1)
        if dkl_orig is None: dkl_orig = dkl_log_probs.detach().clone()
        dkl = handler.kl_factor * torch.nn.functional.kl_div(dkl_orig, dkl_log_probs, log_target=True, reduction='batchmean')
        wd = handler.weight_decay * (torch.norm(delta) / (torch.norm(v_init) ** 2))
        loss = (-1 * log_probs).mean() + dkl + wd
        if verbose: LOGGER.info('epoch=%s loss=%s', step, float(loss.detach().cpu()))
        if step == int(N_optim_steps) - 1: break
        loss.backward(); opt.step()
        max_norm = 4 * v_init.norm()
        if delta.norm() > max_norm:
            with torch.no_grad(): delta[...] = delta * max_norm / delta.norm()
    return delta.detach()

class SM_Method(Enum):
    WIKIPEDIA = 2

def get_free_vram(device=0):
    if not torch.cuda.is_available(): return 0
    if isinstance(device, torch.device): device = str(device)
    if isinstance(device, str):
        if device == 'cpu': return 0
        device = int(device.replace('cuda:', '').replace('cuda', '0') or '0')
    return int(torch.cuda.mem_get_info(device)[0])

def estimate_covariance_batch_size(hidden_dim, max_length, dtype_bytes=2, device=0, vram_fraction=0.15, min_batch=1, max_batch=64):
    """Same estimator as src.common.linalg.estimate_covariance_batch_size."""
    free = get_free_vram(device)
    if free == 0:
        return int(min_batch)
    expected_seq_len = min(int(max_length), 1024)
    per_sample = expected_seq_len * int(hidden_dim) * int(dtype_bytes) * 8
    cov_overhead = int(hidden_dim) * int(hidden_dim) * 4
    available = int(free * float(vram_fraction)) - cov_overhead
    available = max(per_sample, available)
    bs = max(int(min_batch), min(int(max_batch), available // max(per_sample, 1)))
    LOGGER.info(
        'Dynamic covariance batch size: %d  (free_vram=%.1fGB, per_sample=%.1fMB, cov_overhead=%.1fMB, expected_seq=%d, max_length=%d)',
        bs, free / 1e9, per_sample / 1e6, cov_overhead / 1e6, expected_seq_len, int(max_length),
    )
    return int(bs)

def _real_token_rows(hidden_states, attention_mask):
    if attention_mask.dim() != 2:
        raise RuntimeError(f'Expected 2D attention_mask, got rank {attention_mask.dim()}')
    batch_size = int(attention_mask.size(0))
    seq_len = int(attention_mask.size(1))
    hidden_states, _ = _reshape_hidden_states(hidden_states, batch_size, seq_len)
    if tuple(hidden_states.shape[:2]) != tuple(attention_mask.shape):
        raise RuntimeError(f'Activation and attention mask shape mismatch: hidden={tuple(hidden_states.shape[:2])}, mask={tuple(attention_mask.shape)}')
    mask = attention_mask.to(device=hidden_states.device, dtype=torch.bool)
    return hidden_states[mask]

def _accumulate_second_moment_tokens(C, hidden_states, attention_mask):
    """Same token accumulation as src.rome.common._accumulate_second_moment_tokens."""
    rows = _real_token_rows(hidden_states.detach(), attention_mask)
    if rows.numel() == 0:
        return 0
    rows = rows.to(device=C.device, dtype=torch.float32)
    C.addmm_(rows.T, rows)
    return int(rows.size(0))

class _AdaptiveCovarianceBatchSizer:
    """Same adaptive covariance batch sizer as src.rome.common."""
    def __init__(self, initial_batch_size: int, min_batch: int = 1, growth_interval: int = 8):
        self.min_batch = max(1, int(min_batch))
        self.initial_batch_size = max(self.min_batch, int(initial_batch_size))
        self.current_batch_size = self.initial_batch_size
        self.growth_interval = max(1, int(growth_interval))
        self._successful_batches = 0
    def record_oom(self, failed_batch_size: int) -> int:
        failed_batch_size = max(self.min_batch, int(failed_batch_size))
        reduced_size = max(self.min_batch, failed_batch_size // 2)
        self.current_batch_size = min(self.current_batch_size, reduced_size)
        self._successful_batches = 0
        return self.current_batch_size
    def record_success(self) -> int:
        if self.current_batch_size >= self.initial_batch_size:
            self._successful_batches = 0
            return self.current_batch_size
        self._successful_batches += 1
        if self._successful_batches >= self.growth_interval:
            self.current_batch_size = min(self.initial_batch_size, max(self.current_batch_size + 1, self.current_batch_size * 2))
            self._successful_batches = 0
        return self.current_batch_size

def load_second_moment_dataset():
    kwargs = {'token': os.environ['HF_TOKEN']} if os.environ.get('HF_TOKEN') else {}
    raw = load_dataset(SECOND_MOMENT_DATASET_NAME, SECOND_MOMENT_DATASET_CONFIG, **kwargs)
    if isinstance(raw, DatasetDict):
        parts = [raw[split] for split in SECOND_MOMENT_DATASET_SPLITS if split in raw]
        return concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    return raw

def second_moment_wikipedia(handler, N_rounds=1, N_k=None):
    """
    Same computation as src.rome.common.second_moment_wikipedia, with one extension:
    return both inverse covariance and the raw regularized covariance for saving.
    """
    layer_name = handler._layer_name_template.format(handler._layer)
    module = handler._get_module(layer_name)
    hidden_dim = int(handler.hidden_dim)
    model_max_length = getattr(handler.model.config, 'n_positions', getattr(handler.model.config, 'max_position_embeddings', 1024))
    max_length_cap = getattr(handler.cfg.model, 'second_moment_max_length', None)
    max_length = int(model_max_length if max_length_cap is None else min(int(max_length_cap), model_max_length))
    module_device = handler.get_module_device(layer_name) if getattr(handler, 'is_multi_gpu', False) else handler.device
    C = torch.zeros(hidden_dim, hidden_dim, dtype=torch.float32, device=module_device)
    total_tokens = 0
    current_attention_mask = None
    def hook(_, inp, out):
        nonlocal C, total_tokens, current_attention_mask
        if current_attention_mask is None:
            raise RuntimeError('Missing attention mask while accumulating covariance')
        hidden_states = inp[0] if isinstance(inp, tuple) else inp
        total_tokens += _accumulate_second_moment_tokens(C, hidden_states, current_attention_mask)
        return out
    handle = module.register_forward_hook(hook)
    n_samples = int(N_rounds) * int(N_k) if N_rounds and N_k else 5000
    dtype_bytes = 2 if handler.dtype in (torch.float16, torch.bfloat16) else 4
    batch_size = estimate_covariance_batch_size(hidden_dim=hidden_dim, max_length=max_length, dtype_bytes=dtype_bytes, device=module_device)
    batch_mode_raw = getattr(handler.cfg.model, 'second_moment_batch_size_mode', SECOND_MOMENT_BATCH_SIZE_MODE)
    batch_mode = str(batch_mode_raw).strip().lower()
    manual_batch_size = getattr(handler.cfg.model, 'second_moment_batch_size', SECOND_MOMENT_BATCH_SIZE)
    adaptive_batch_enabled = False
    if batch_mode in ('manual', 'fixed', 'static'):
        if manual_batch_size is None:
            raise ValueError("second_moment_batch_size must be set when second_moment_batch_size_mode is 'manual'")
        batch_size = max(1, int(manual_batch_size))
        LOGGER.info('Using manual covariance batch size override: %d', batch_size)
    elif batch_mode in ('dynamic', 'auto'):
        if batch_mode == 'auto' and manual_batch_size is not None:
            batch_size = max(1, int(manual_batch_size))
            LOGGER.info('Using manual covariance batch size override: %d (mode=auto)', batch_size)
        else:
            adaptive_batch_enabled = True
            LOGGER.info('Using dynamic covariance batch size estimate: %d', batch_size)
    else:
        raise ValueError(f"Invalid second_moment_batch_size_mode: {batch_mode_raw!r}. Expected one of: auto, dynamic, manual.")
    batch_sizer = _AdaptiveCovarianceBatchSizer(batch_size)
    LOGGER.info('Starting covariance computation: %d samples, batch_size=%d, max_length=%d', n_samples, batch_size, max_length)
    ds = load_second_moment_dataset()
    if getattr(handler, 'is_multi_gpu', False):
        try:
            input_module_name = handler._corrupt_layer_name_template
            if '{}' in input_module_name:
                input_module_name = input_module_name.format(0)
            input_device = handler.get_module_device(input_module_name)
        except Exception:
            input_device = next(handler.model.parameters()).device
    else:
        input_device = handler.device
    min_text_length = int(getattr(handler.cfg.model, 'second_moment_min_text_length', SECOND_MOMENT_MIN_TEXT_LENGTH))
    processed = 0
    processed_batches = 0
    clear_cache_every = int(getattr(handler.cfg.model, 'second_moment_clear_cache_every', SECOND_MOMENT_CLEAR_CACHE_EVERY) or 0)
    batch_texts = []
    def process_text_batch(text_batch):
        nonlocal processed, batch_size, processed_batches, current_attention_mask
        if not text_batch:
            return
        queue = [text_batch]
        while queue:
            chunk = queue.pop(0)
            tokens = None
            try:
                tokens = handler.tokenizer(chunk, return_tensors='pt', truncation=True, max_length=max_length, padding=True)
                current_attention_mask = tokens.attention_mask
                handler.model(tokens.input_ids.to(input_device), attention_mask=tokens.attention_mask.to(input_device), use_cache=False)
                processed += len(chunk)
                processed_batches += 1
                if adaptive_batch_enabled:
                    old_batch_size = batch_size
                    batch_size = batch_sizer.record_success()
                    if batch_size != old_batch_size:
                        LOGGER.info('Increasing covariance batch size after successful batches: %d -> %d', old_batch_size, batch_size)
                if clear_cache_every > 0 and processed_batches % clear_cache_every == 0 and torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except torch.cuda.OutOfMemoryError:
                LOGGER.warning('OOM during covariance computation (chunk=%d)', len(chunk))
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                if len(chunk) <= 1:
                    LOGGER.warning('Skipping sample that causes OOM even at batch_size=1')
                    continue
                if adaptive_batch_enabled:
                    old_batch_size = batch_size
                    batch_size = batch_sizer.record_oom(len(chunk))
                    if batch_size != old_batch_size:
                        LOGGER.warning('Reduced covariance batch size: %d -> %d', old_batch_size, batch_size)
                else:
                    LOGGER.warning('Splitting covariance chunk while fixed batch size remains %d', batch_size)
                midpoint = max(1, len(chunk) // 2)
                queue.insert(0, chunk[midpoint:])
                queue.insert(0, chunk[:midpoint])
            except Exception as e:
                LOGGER.warning(e)
            finally:
                current_attention_mask = None
                if tokens is not None:
                    del tokens
    try:
        with torch.no_grad(), tqdm(total=n_samples, desc='Computing covariance', mininterval=1.0) as pbar:
            for sample in ds:
                if processed >= n_samples:
                    break
                text = sample.get('text', '')
                if len(text.strip()) < min_text_length:
                    continue
                batch_texts.append(text)
                remaining = n_samples - processed
                if remaining <= 0:
                    break
                if len(batch_texts) >= batch_size or len(batch_texts) >= remaining:
                    take_n = min(len(batch_texts), remaining)
                    old_processed = processed
                    process_text_batch(batch_texts[:take_n])
                    pbar.update(max(0, processed - old_processed))
                    batch_texts = []
            if batch_texts and processed < n_samples:
                remaining = n_samples - processed
                old_processed = processed
                process_text_batch(batch_texts[:remaining])
                pbar.update(max(0, processed - old_processed))
    finally:
        handle.remove()
    if processed < n_samples:
        raise RuntimeError(f'Covariance sampling incomplete: processed {processed} samples out of target {n_samples}.')
    if total_tokens == 0:
        raise ValueError('No samples processed for covariance!')
    LOGGER.info('Processed %d samples and %d tokens, computing inverse covariance...', processed, total_tokens)
    cov = C / total_tokens
    cov += 1e-5 * torch.eye(hidden_dim, device=cov.device)
    LOGGER.info('Inverting %dx%d covariance matrix on device %s...', hidden_dim, hidden_dim, cov.device)
    inv_cov = torch.linalg.inv(cov)
    return inv_cov.to('cpu'), cov.detach().to('cpu')

def compute_second_moment(handler, N_rounds=1, N_k=None, method=SM_Method.WIKIPEDIA):
    inv_cov, raw_cov = second_moment_wikipedia(handler, N_rounds=N_rounds, N_k=N_k)
    return inv_cov, raw_cov, int(N_rounds) * int(N_k or SECOND_MOMENT_TARGET_SAMPLES), method

def non_conflicting_path(path):
    path = Path(path)
    if not path.exists():
        return path
    for idx in range(1, 10000):
        candidate = path.with_name(f'{path.stem}_{idx}{path.suffix}')
        if not candidate.exists():
            return candidate
    raise RuntimeError(f'Could not find non-conflicting path for {path}')

def save_second_moment_matrices(handler, inv_cov, raw_cov, count, method):
    model_slug = handler.cfg.model.name.replace('/', '_')
    stem = f"{model_slug}_{handler._layer}_{method}_{int(count)}"
    inv_dir = Path(handler.second_moment_dir); inv_dir.mkdir(parents=True, exist_ok=True)
    inv_path = non_conflicting_path(inv_dir / f"{stem}.pt")
    torch.save(inv_cov.detach().cpu(), inv_path)
    raw_path = None
    if SAVE_RAW_COVARIANCE and raw_cov is not None:
        raw_dir = Path(RAW_COVARIANCE_DIR); raw_dir.mkdir(parents=True, exist_ok=True)
        raw_path = non_conflicting_path(raw_dir / f"{model_slug}_{handler._layer}_raw_covariance_{method}_{int(count)}.pt")
        torch.save(raw_cov.detach().cpu(), raw_path)
    handler.second_moment_path = str(inv_path)
    handler.last_second_moment_paths = {
        'inverse_covariance_path': str(inv_path),
        'raw_covariance_path': str(raw_path) if raw_path is not None else None,
    }
    return inv_path, raw_path

def get_second_moment(handler):
    target_device = handler.get_module_device(handler._layer_name_template.format(handler._layer))
    cached = getattr(handler, '_second_moment_cache', None)
    if cached is not None and handler.second_moment_path and cached[0] == str(handler.second_moment_path):
        matrix = cached[1].to(dtype=handler.dtype, device=target_device)
        handler._second_moment_cache = (cached[0], matrix)
        return matrix
    status = second_moment_status(handler)
    if status['available']:
        path = Path(status['inverse_covariance_paths'][0])
        handler.second_moment_path = str(path)
        cached = getattr(handler, '_second_moment_cache', None)
        matrix = cached[1] if cached is not None and cached[0] == str(path) else _load_second_moment_matrix(path, handler.hidden_dim)
        matrix = matrix.to(dtype=handler.dtype, device=target_device)
        handler._second_moment_cache = (str(path), matrix)
        return matrix
    if not SECOND_MOMENT_ALLOW_AUTOCOMPUTE: raise FileNotFoundError(f'Missing second moment for {handler.model_config} layer={handler._layer}')
    inv_cov, raw_cov, count, method = compute_second_moment(handler, N_rounds=1, N_k=SECOND_MOMENT_TARGET_SAMPLES)
    save_second_moment_matrices(handler, inv_cov, raw_cov, count, method)
    return inv_cov.to(handler.get_module_device(handler._layer_name_template.format(handler._layer)))

def insert_kv(handler, k, delta):
    layer_name = handler._layer_name_template.format(handler._layer); module = handler._get_module(layer_name); layer_device = handler.get_module_device(layer_name)
    old_W = module.weight.clone(); W = old_W; transposed = False
    if W.shape[0] != k.shape[0]: W = W.T; transposed = True
    residual_mult = float(getattr(handler.model.config, 'residual_multiplier', 1.0)); delta_scale = (1.0 / residual_mult) if 0 < residual_mult < 1.0 else 1.0
    inv_cov = get_second_moment(handler).to(handler.dtype).to(layer_device); k = k.to(layer_device); scaled_delta = (delta * delta_scale).to(layer_device)
    left = (inv_cov @ k.unsqueeze(1)).squeeze(); left = left / left.norm(); right = scaled_delta / torch.dot(k, left)
    update = left.unsqueeze(1) @ right.unsqueeze(0)
    try: new_W = W + update
    except Exception: new_W = W + update.T
    if transposed: new_W = new_W.T
    module.weight = torch.nn.Parameter(new_W.to(dtype=module.weight.dtype, device=module.weight.device))
    return new_W.to(handler.dtype), old_W, update

def _target_ids(tokenizer, text):
    ids = tokenizer(f' {text}', add_special_tokens=False)['input_ids'] or tokenizer(f' {text}')['input_ids']
    if isinstance(ids, torch.Tensor): ids = ids.tolist()
    bos = getattr(tokenizer, 'bos_token_id', None)
    if bos is not None and len(ids) > 1 and ids[0] == bos: ids = ids[1:]
    return list(ids)

def test_batch_prediction_rome(model, tokenizer, prefixes, target_new, target_true, device, batch_size=8):
    new_ids, true_ids = _target_ids(tokenizer, target_new), _target_ids(tokenizer, target_true)
    choices, lengths, results = (new_ids, true_ids), (len(new_ids), len(true_ids)), []
    for start in range(0, len(prefixes), batch_size):
        chunk = list(prefixes[start:start + batch_size]); prefix_lengths = [len(x) for x in tokenizer(chunk)['input_ids']]
        tokens = tokenizer([f'{p} {s}' for p in chunk for s in (target_new, target_true)], padding=True, return_tensors='pt').to(device)
        pad_offsets = (tokens['attention_mask'].cumsum(dim=1) == 0).sum(dim=1).tolist()
        with torch.no_grad(): logits = model(**tokens, use_cache=False).logits
        losses = np.zeros((logits.size(0),), dtype=np.float32)
        for row in range(logits.size(0)):
            choice = row % 2
            for token_i, token_id in enumerate(choices[choice]):
                pos = pad_offsets[row] + prefix_lengths[row // 2] + token_i - 1
                losses[row] += -torch.nn.functional.log_softmax(logits[row, pos, :], dim=0)[token_id].item()
            losses[row] /= lengths[choice]
        for row in range(0, len(losses), 2): results.append({'target_new': losses[row].item(), 'target_true': losses[row + 1].item()})
        del tokens, logits
    return results

def compute_rome_metrics_notebook(handler, prompt_text, target_new, target_true, paraphrase_prompts=None, neighborhood_prompts=None):
    paraphrases, neighborhoods = list(paraphrase_prompts or []), list(neighborhood_prompts or [])
    probs = test_batch_prediction_rome(handler.model, handler.tokenizer, [prompt_text, *paraphrases, *neighborhoods], target_new, target_true, handler.device)
    rewrite = probs[0]; para = probs[1:1 + len(paraphrases)]; neigh = probs[1 + len(paraphrases):]
    efficacy = 1.0 if rewrite['target_new'] < rewrite['target_true'] else 0.0
    magnitude = math.exp(-rewrite['target_new']) - math.exp(-rewrite['target_true'])
    paraphrase_score = sum(v['target_new'] < v['target_true'] for v in para) / len(para) if para else None
    neighborhood_score = sum(v['target_true'] < v['target_new'] for v in neigh) / len(neigh) if neigh else None
    comps = [x for x in (efficacy, paraphrase_score, neighborhood_score) if x is not None]
    overall = len(comps) / sum(1.0 / x for x in comps) if comps and all(x > 0 for x in comps) else 0.0
    return {'efficacy_score': efficacy, 'efficacy_magnitude': magnitude, 'paraphrase_score': paraphrase_score, 'neighborhood_score': neighborhood_score, 'overall_score': overall, 'rewrite_nll': rewrite, 'paraphrase_nll': para, 'neighborhood_nll': neigh}
print('Self-contained ROME primitives loaded')


## 12. Covariance and ROME Benchmark Helpers

Helper functions used by the final two optional cells.


In [ ]:
def counterfact_row_to_rome_case(row, dataset_index):
    rw = row.get('requested_rewrite', row)
    target_new = rw['target_new']['str'] if isinstance(rw.get('target_new'), dict) else str(rw.get('target_new'))
    target_true = rw['target_true']['str'] if isinstance(rw.get('target_true'), dict) else str(rw.get('target_true'))
    return {'dataset_index': int(dataset_index), 'case_id': int(row.get('case_id', dataset_index)), 'relation_id': rw.get('relation_id', ''), 'subject': rw['subject'], 'target_new_str': target_new, 'target_true_str': target_true, 'fact_tuple': (rw['prompt'], rw['subject'], ' ' + target_new, ' ' + target_true), 'prompt_text': rw['prompt'].format(rw['subject']), 'paraphrase_prompts': row.get('paraphrase_prompts', []) or [], 'neighborhood_prompts': row.get('neighborhood_prompts', []) or []}

def _path_matches_model_layer(path, model_slug, layer):
    path = Path(path)
    return (
        path.is_file()
        and path.suffix.lower() in {'.pt', '.npz'}
        and path.name.startswith(f'{model_slug}_{int(layer)}_')
        and 'raw_covariance' not in path.name.lower()
    )

def _second_moment_sample_count(path):
    match = re.search(r'_(\d+)(?:_\d+)?\.(?:pt|npz)$', Path(path).name)
    return int(match.group(1)) if match else -1

def _load_second_moment_matrix(path, expected_dim):
    path = Path(path)
    if path.suffix.lower() == '.npz':
        with np.load(path) as archive:
            if 'mom2.mom2' not in archive:
                raise KeyError("missing 'mom2.mom2'")
            raw = archive['mom2.mom2']
            shape = tuple(raw.shape)
            if shape != (int(expected_dim), int(expected_dim)):
                raise ValueError(f'expected {(int(expected_dim), int(expected_dim))}, got {shape}')
            matrix = torch.from_numpy(np.asarray(raw)).to(torch.float32).inverse()
    else:
        try:
            matrix = torch.load(path, map_location='cpu', weights_only=True)
        except TypeError:
            matrix = torch.load(path, map_location='cpu')
        if not torch.is_tensor(matrix):
            raise TypeError(f'expected a tensor, got {type(matrix).__name__}')
        if tuple(matrix.shape) != (int(expected_dim), int(expected_dim)):
            raise ValueError(
                f'expected {(int(expected_dim), int(expected_dim))}, got {tuple(matrix.shape)}'
            )
        matrix = matrix.to(torch.float32)
    return matrix

def second_moment_status(handler):
    model_slug = handler.cfg.model.name.replace('/', '_')
    layer = int(handler._layer)
    configured_path = Path(handler.second_moment_path) if handler.second_moment_path else None
    candidates = []
    if configured_path is not None:
        if _path_matches_model_layer(configured_path, model_slug, layer):
            candidates.append(configured_path)
        elif configured_path.exists():
            print(
                f'Ignoring configured second moment for a different model/layer: '
                f'active={model_slug} L{layer}, path={configured_path}'
            )
    sm_dir = Path(handler.second_moment_dir)
    if sm_dir.exists():
        candidates.extend(sm_dir.glob(f'{model_slug}_{layer}_*.pt'))
        candidates.extend(sm_dir.glob(f'{model_slug}_{layer}_*.npz'))
    candidates = [
        path for path in set(Path(path) for path in candidates)
        if _path_matches_model_layer(path, model_slug, layer)
    ]
    candidates.sort(
        key=lambda path: (
            configured_path is not None and path.resolve() == configured_path.resolve(),
            path.suffix.lower() == '.pt',
            _second_moment_sample_count(path),
            path.stat().st_mtime,
        ),
        reverse=True,
    )
    inverse_paths = []
    invalid_paths = []
    for path in candidates:
        try:
            cached = getattr(handler, '_second_moment_cache', None)
            if cached is not None and cached[0] == str(path):
                matrix = cached[1]
            else:
                matrix = _load_second_moment_matrix(path, handler.hidden_dim)
                handler._second_moment_cache = (str(path), matrix)
            inverse_paths.append(path)
            break
        except Exception as exc:
            invalid_paths.append({'path': str(path), 'reason': f'{type(exc).__name__}: {exc}'})
    raw_dir = Path(RAW_COVARIANCE_DIR)
    raw_paths = sorted(raw_dir.glob(f"{model_slug}_{layer}_raw_covariance_*_*.pt")) if raw_dir.exists() else []
    paths = [*inverse_paths, *raw_paths]
    return {
        'available': bool(inverse_paths),
        'raw_covariance_available': bool(raw_paths),
        'paths': [str(p) for p in paths],
        'inverse_covariance_paths': [str(p) for p in inverse_paths],
        'invalid_inverse_covariance_paths': invalid_paths,
        'raw_covariance_paths': [str(p) for p in raw_paths],
        'model_slug': model_slug,
        'layer': layer,
        'expected_matrix_dim': int(handler.hidden_dim),
        'module': handler._layer_name_template.format(handler._layer),
    }

def compute_and_save_second_moment_for_model(model_config, layer_override=None):
    handler = NotebookRomeHandler(model_config, layer_override=layer_override)
    try:
        before = second_moment_status(handler)
        if before['available']:
            handler.second_moment_path = before['inverse_covariance_paths'][0]
            after = dict(before)
            after.update({
                'available': True,
                'reused_existing': True,
                'computed_now': False,
                'inverse_covariance_path': before['inverse_covariance_paths'][0],
                'raw_covariance_path': before['raw_covariance_paths'][0] if before.get('raw_covariance_paths') else None,
            })
            print(f"{model_config}: reusing existing covariance for layer {handler._layer}: {after['inverse_covariance_path']}")
            return {'model_config': model_config, 'graph_selected_benchmark_layer': None if layer_override is None else int(layer_override), 'status_before': before, 'status_after': after}
        inv_cov, raw_cov, count, method = compute_second_moment(handler, N_rounds=1, N_k=SECOND_MOMENT_TARGET_SAMPLES)
        inv_path, raw_path = save_second_moment_matrices(handler, inv_cov, raw_cov, count, method)
        after = second_moment_status(handler)
        after.update({
            'available': True,
            'reused_existing': False,
            'computed_now': True,
            'count': int(count),
            'method': str(method),
            'inverse_covariance_path': str(inv_path),
            'raw_covariance_path': str(raw_path) if raw_path is not None else None,
        })
        return {'model_config': model_config, 'graph_selected_benchmark_layer': None if layer_override is None else int(layer_override), 'status_before': before, 'status_after': after}
    finally:
        del handler; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

def run_rome_benchmark_for_model(model_config, dataset, *, n_edits=100, start_idx=0, out_dir=None, layer_override=None, second_moment_path_override=None, layer_source=None, target_evaluated=False, max_attempts=None, zero_evaluated_abort_after=None):
    handler = NotebookRomeHandler(model_config, layer_override=layer_override)
    if second_moment_path_override is not None:
        handler.second_moment_path = str(second_moment_path_override)
    out_dir = Path(out_dir or (OUT_ROOT / f'{model_config}_{RUN_TIMESTAMP}' / f'rome_benchmark_layer_{handler._layer}')); out_dir.mkdir(parents=True, exist_ok=True)
    layer_name = handler._layer_name_template.format(handler._layer); sm_status = second_moment_status(handler)
    (out_dir / 'second_moment_status.json').write_text(json.dumps(json_safe(sm_status), indent=2))
    if not sm_status['available'] and not SECOND_MOMENT_ALLOW_AUTOCOMPUTE: raise FileNotFoundError(f'Missing second moment for {model_config} layer={handler._layer}; run the covariance cell or enable SECOND_MOMENT_ALLOW_AUTOCOMPUTE')

    requested = int(n_edits)
    dataset_len = int(len(dataset))
    start_idx = int(start_idx)
    if max_attempts is None:
        max_attempts = requested * int(ROME_MAX_ATTEMPT_MULTIPLIER) if target_evaluated else requested
    max_attempts = max(requested, int(max_attempts))
    max_dataset_index = min(dataset_len, start_idx + max_attempts)

    cases, tested, skipped = [], 0, 0
    es_scores, em_scores, ps_scores, ns_scores, s_scores = [], [], [], [], []
    error_counts, error_examples = {}, []
    stopped_reason = 'dataset_exhausted'
    progress_total = requested if target_evaluated else min(requested, max_dataset_index - start_idx)
    progress = tqdm(total=progress_total, desc=f'ROME {model_config} L{handler._layer}')
    try:
        idx = start_idx
        while idx < max_dataset_index:
            n_evaluated = tested - skipped
            if target_evaluated:
                if n_evaluated >= requested:
                    stopped_reason = 'target_evaluated_reached'
                    break
            elif tested >= requested:
                stopped_reason = 'target_attempts_reached'
                break
            if zero_evaluated_abort_after is not None and tested >= int(zero_evaluated_abort_after) and n_evaluated == 0:
                stopped_reason = f'zero_evaluated_abort_after_{zero_evaluated_abort_after}'
                break

            case = counterfact_row_to_rome_case(dataset[int(idx)], idx); idx += 1; tested += 1
            module = handler._get_module(layer_name); original_weight = module.weight.detach().clone(); metrics = None; error = None
            try:
                k = gather_k(handler, case['fact_tuple'], N=resolve_rome_sample_count(handler, 'k_N'))
                delta = optimize_v(handler, case['fact_tuple'], N_prompts=resolve_rome_sample_count(handler, 'v_N'), N_optim_steps=handler.epochs, verbose=ROME_OPTIMIZER_VERBOSE)
                if delta is None: raise RuntimeError('optimize_v returned None')
                insert_kv(handler, k, delta)
                metrics = compute_rome_metrics_notebook(handler, case['prompt_text'], case['target_new_str'], case['target_true_str'], case['paraphrase_prompts'], case['neighborhood_prompts'])
                es_scores.append(metrics['efficacy_score']); em_scores.append(metrics['efficacy_magnitude']); s_scores.append(metrics['overall_score'])
                if metrics['paraphrase_score'] is not None: ps_scores.append(metrics['paraphrase_score'])
                if metrics['neighborhood_score'] is not None: ns_scores.append(metrics['neighborhood_score'])
            except Exception as exc:
                skipped += 1
                error = f'{type(exc).__name__}: {exc}'
                error_counts[error] = int(error_counts.get(error, 0)) + 1
                if len(error_examples) < 5:
                    error_examples.append({'case_id': case['case_id'], 'relation_id': case['relation_id'], 'subject': case['subject'], 'error': error})
            finally:
                handler._get_module(layer_name).weight = torch.nn.Parameter(original_weight); handler.remove_hooks()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
            row = {'case_id': case['case_id'], 'dataset_index': case['dataset_index'], 'relation_id': case['relation_id'], 'subject': case['subject'], 'target_new': case['target_new_str'], 'target_true': case['target_true_str'], 'error': error}
            if metrics is not None: row.update(metrics)
            cases.append(row)
            if metrics is not None or not target_evaluated:
                progress.update(1)
            if target_evaluated:
                progress.set_postfix(evaluated=tested-skipped, tested=tested, skipped=skipped)
    finally:
        progress.close()

    final_sm_status = second_moment_status(handler)
    (out_dir / 'second_moment_status.json').write_text(json.dumps(json_safe(final_sm_status), indent=2))
    n_evaluated = int(tested - skipped)
    summary = {
        'model_key': model_config,
        'model_name': handler.cfg.model.name,
        'layer': int(handler._layer),
        'layer_source': layer_source,
        'configured_layer_reference': int(handler.configured_layer),
        'graph_selected_benchmark_layer': None if layer_override is None else int(layer_override),
        'config_layer_used_for_selection': False,
        'layer_module': layer_name,
        'requested_evaluated_edits': int(requested),
        'target_evaluated': bool(target_evaluated),
        'max_attempts': int(max_attempts),
        'stopped_reason': stopped_reason,
        'tested': int(tested),
        'skipped': int(skipped),
        'n_evaluated': n_evaluated,
        'mean_efficacy_score': float(np.mean(es_scores)) if es_scores else 0.0,
        'mean_efficacy_magnitude': float(np.mean(em_scores)) if em_scores else 0.0,
        'mean_paraphrase_score': float(np.mean(ps_scores)) if ps_scores else 0.0,
        'mean_neighborhood_score': float(np.mean(ns_scores)) if ns_scores else 0.0,
        'mean_overall_score': float(np.mean(s_scores)) if s_scores else 0.0,
        'inverse_covariance_paths': ';'.join(final_sm_status.get('inverse_covariance_paths', [])),
        'raw_covariance_paths': ';'.join(final_sm_status.get('raw_covariance_paths', [])),
        'first_error': error_examples[0]['error'] if error_examples else None,
        'error_counts': error_counts,
        'error_examples': error_examples,
    }
    with (out_dir / 'rome_benchmark_cases.jsonl').open('w') as f:
        for case in cases: f.write(json.dumps(json_safe(case)) + '\n')
    pd.DataFrame(cases).to_csv(out_dir / 'rome_benchmark_cases.csv', index=False)
    (out_dir / 'rome_benchmark_summary.json').write_text(json.dumps(json_safe(summary), indent=2))
    pd.DataFrame([summary]).to_csv(out_dir / 'rome_benchmark_summary.csv', index=False)
    del handler; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {'summary': summary, 'cases': cases, 'out_dir': str(out_dir)}
print('ROME benchmark helpers loaded')


## 13. Optional: Compute Covariance / Second-Moment Matrices

This cell ensures the ROME inverse covariance for each newly selected trace layer. Before computing, it searches `SECOND_MOMENT_DIR` for an exact model-and-layer match and reuses a valid existing `.pt` inverse matrix or legacy `.npz` moment file. New computations also save the raw regularized covariance for audit. If the selected layer matches the config layer, this separate stage skips it.


In [ ]:
second_moment_results = []

if RUN_SECOND_MOMENT:
    for model_config in ROME_BENCHMARK_MODELS:
        trace = trace_by_model.get(model_config)
        layer, reason = trace_layer_for_downstream(model_config, trace)
        if layer is None:
            msg = f'{model_config}: covariance skipped: {reason}'
            print(msg, flush=True)
            second_moment_results.append(skipped_result(model_config, 'second_moment', reason))
            continue
        print(f'{model_config}: ensuring covariance for newly selected layer {layer} ({reason})', flush=True)
        second_moment_results.append(compute_and_save_second_moment_for_model(model_config, layer_override=layer))
else:
    print('RUN_SECOND_MOMENT=False; skipping covariance computation.')

if second_moment_results:
    second_moment_rows = []
    for item in second_moment_results:
        if item.get('skipped'):
            second_moment_rows.append(item)
        else:
            second_moment_rows.append({'model_config': item['model_config'], **item['status_after']})
    second_moment_summary = pd.DataFrame(second_moment_rows)
    display(second_moment_summary)
    if SAVE:
        path = OUT_ROOT / f'second_moment_summary_{RUN_TIMESTAMP}.json'
        path.write_text(json.dumps(json_safe(second_moment_results), indent=2))
        print(f'Wrote second moment summary to {path}')


## 14. Optional: Run ROME Benchmark

This cell runs the self-contained ROME benchmark. It resolves the exact selected model/layer covariance again, so it reuses an existing compatible file even when the separate covariance cell was skipped or the notebook kernel was resumed.


In [ ]:
def saved_inverse_covariance_path_for_model(model_config, layer=None):
    for item in reversed(second_moment_results if 'second_moment_results' in globals() else []):
        if item.get('model_config') != model_config or item.get('skipped'):
            continue
        status = item.get('status_after', {})
        if layer is not None and int(status.get('layer', -1)) != int(layer):
            continue
        path = status.get('inverse_covariance_path')
        if path and Path(path).is_file():
            return path
        paths = status.get('inverse_covariance_paths') or []
        if paths and Path(paths[0]).is_file():
            return paths[0]
    return None

def ensure_covariance_for_rome(model_config, layer):
    sm_path = saved_inverse_covariance_path_for_model(model_config, layer=layer)
    if sm_path:
        return sm_path
    result = compute_and_save_second_moment_for_model(model_config, layer_override=layer)
    second_moment_results.append(result)
    sm_path = saved_inverse_covariance_path_for_model(model_config, layer=layer)
    if sm_path is None:
        raise FileNotFoundError(f'No compatible second moment found or created for {model_config} layer {layer}')
    return sm_path

def run_rome_layer(model_config, layer, source, *, target_evaluated=True, zero_abort=None, retry_of_layer=None, retry_reason=None):
    sm_path = ensure_covariance_for_rome(model_config, layer)
    print(f'{model_config}: running ROME benchmark, requested_evaluated={ROME_BENCHMARK_NUM_EDITS}, layer={layer}, covariance={sm_path}, source={source}', flush=True)
    result = run_rome_benchmark_for_model(
        model_config,
        dataset,
        n_edits=ROME_BENCHMARK_NUM_EDITS,
        start_idx=ROME_BENCHMARK_START_IDX,
        layer_override=layer,
        second_moment_path_override=sm_path,
        layer_source=source,
        target_evaluated=target_evaluated,
        zero_evaluated_abort_after=zero_abort,
    )
    if retry_of_layer is not None:
        result['summary']['retry_of_layer'] = int(retry_of_layer)
        result['summary']['retry_reason'] = retry_reason
    return result

rome_benchmark_results = []

if RUN_ROME_BENCHMARK:
    for model_config in ROME_BENCHMARK_MODELS:
        trace = trace_by_model.get(model_config)
        layer, reason = trace_layer_for_downstream(model_config, trace)
        if layer is None:
            config_layer = config_reference_layer_for_model(model_config, trace)
            if ROME_RUN_CONFIG_LAYER_ON_TRACE_SKIP and config_layer is not None:
                print(f'{model_config}: trace layer unavailable/skipped ({reason}); running config/reference layer {config_layer} for ROME benchmark only.', flush=True)
                result = run_rome_layer(model_config, config_layer, f'config_reference_benchmark_fallback:{reason}', target_evaluated=True)
                rome_benchmark_results.append(result)
            else:
                msg = f'{model_config}: ROME benchmark skipped: {reason}'
                print(msg, flush=True)
                rome_benchmark_results.append({'summary': {'model_key': model_config, 'skipped': True, 'skip_reason': reason, 'n_evaluated': 0}, 'cases': [], 'out_dir': None})
            continue

        result = run_rome_layer(
            model_config,
            layer,
            reason,
            target_evaluated=ROME_FINAL_TARGET_EVALUATED_EDITS,
            zero_abort=ROME_PRIMARY_ZERO_EVAL_ABORT_AFTER,
        )
        rome_benchmark_results.append(result)

        retry, config_layer, retry_reason = should_retry_config_layer_after_failed_rome(model_config, trace, layer, result)
        if retry:
            print(f'{model_config}: primary ROME layer {layer} evaluated zero cases; retrying config/reference layer {config_layer} ({retry_reason})', flush=True)
            retry_result = run_rome_layer(
                model_config,
                config_layer,
                retry_reason,
                target_evaluated=True,
                zero_abort=None,
                retry_of_layer=layer,
                retry_reason=retry_reason,
            )
            rome_benchmark_results.append(retry_result)
else:
    print('RUN_ROME_BENCHMARK=False; skipping ROME benchmark.')

if rome_benchmark_results:
    rome_summary = pd.DataFrame([r['summary'] for r in rome_benchmark_results])
    display(rome_summary)
    if SAVE:
        csv_path = OUT_ROOT / f'rome_benchmark_summary_{RUN_TIMESTAMP}.csv'
        json_path = OUT_ROOT / f'rome_benchmark_all_{RUN_TIMESTAMP}.json'
        rome_summary.to_csv(csv_path, index=False)
        json_path.write_text(json.dumps(json_safe({'results': rome_benchmark_results}), indent=2))
        print(f'Wrote ROME benchmark summary to {csv_path}')
